# Reproducible original PriceTracker baseline

The original fast-renderer TypeScript sources, compiled JavaScript, npm dependencies and
third-party licenses are embedded in this notebook. No PriceTracker checkout or npm installation
is needed. Node.js executes the original code; Python evaluates results and creates Excel reports.

Default methods: `original_fast` (all page products), `original_fast_query` (original query filter).
Both reproduce the methods in `original_comparison.xlsx`. B1/B2 below are retained only as optional
historical controls; they are not the default baseline for the paper.

Reference environment: Python 3.13.13, Node.js 24.6.0. Place the notebook next to `industrial/`
and `synthetic/`, then Restart Kernel → Run All. Results: `baseline_results/determined/`.


In [1]:
%pip install "pandas==3.0.6" "beautifulsoup4==4.15.0" "lxml==6.1.3" "openpyxl==3.1.5"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from __future__ import annotations

from pathlib import Path
from urllib.parse import urljoin, urlsplit
from difflib import SequenceMatcher
import hashlib
import html as html_std
import importlib.util
import json
import math
import re
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

from datetime import datetime, timezone
import uuid

from time import perf_counter
import traceback


## 1. Configuration

The benchmark is self-contained. Put the notebook in the benchmark directory with this layout:

```text
benchmark/
├─ baseline_determined.ipynb
├─ config/requirements-benchmark.lock.txt     # pinned environment
├─ scripts/parse_logs_to_excel_v2.py          # optional parity check only
├─ synthetic/
│  ├─ html/
│  └─ [dind.xlsx | synthetic_ground_truth.xlsx | synthetic_ground_truth.csv | ...]
└─ industrial/
   ├─ html/
   └─ dind.xlsx
```

The notebook automatically finds the **nearest directory containing both `synthetic/html/` and `industrial/html/`**. Normally this is simply the notebook's working directory.

It does not depend on environment variables or on the original PriceTracker repository. If `scripts/parse_logs_to_excel_v2.py` is present, it is used only for an optional runtime parity check; the evaluation contract is already embedded in the notebook.

Outputs are written to `baseline_results/determined/` next to the datasets.

Synthetic data has two separate splits sharing synthetic/html: synthetic uses
synthetic_ground_truth.csv + synthetic_pages.csv (20 clean pages); robustness
uses robustness_ground_truth.csv + robustness_pages.csv (200 mutations).
RUN_DATASETS=None runs everything. RUN_DATASETS=("synthetic", "robustness")
reruns those splits and retains verified industrial results in the combined files.


In [3]:
# ----------------------------------------
# Self-contained benchmark configuration
# ----------------------------------------

def _find_benchmark_root(start: Path | None = None) -> Path:
    """Find the nearest folder that contains both benchmark HTML directories."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "synthetic" / "html").is_dir()
            and (candidate / "industrial" / "html").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find a benchmark root containing both "
        "'synthetic/html/' and 'industrial/html/'. "
        "Place this notebook next to the synthetic/ and industrial/ folders "
        "or start Jupyter from that directory."
    )


BENCHMARK_ROOT = _find_benchmark_root()

DATASETS = {
    "synthetic": {
        "root": BENCHMARK_ROOT / "synthetic",
        "html_dir": BENCHMARK_ROOT / "synthetic" / "html",
        "reference_file": None,  # auto-discovered under synthetic/
    },
    "robustness": {
        "root": BENCHMARK_ROOT / "synthetic",
        "html_dir": BENCHMARK_ROOT / "synthetic" / "html",
        "reference_file": BENCHMARK_ROOT / "synthetic" / "robustness_ground_truth.csv",
    },
    "industrial": {
        "root": BENCHMARK_ROOT / "industrial",
        "html_dir": BENCHMARK_ROOT / "industrial" / "html",
        "reference_file": None,  # auto-discovered under industrial/
    },
}

OUTPUT_DIR = BENCHMARK_ROOT / "baseline_results" / "determined"

# Synthetic v2: rerun the two updated splits. Set None to run all datasets.
# 20 clean pages / 420 products; 200 mutations / 4200 products.
# Other datasets' verified saved results are retained in the combined CSVs.
RUN_DATASETS = ("synthetic", "robustness")

# Exact original parser modes used in original_comparison.xlsx.
VARIANTS = ("original_fast", "original_fast_query")
# Optional historical controls: append "b1_jsonld", "b2_css" or "b3_regex".
ORIGINAL_RUNTIME_DIR = BENCHMARK_ROOT / ".benchmark_runtime"
# False always performs fresh extraction. True resumes verified input/code checkpoints.
RESUME_EXTRACTION = False

NAME_THRESHOLD = 0.85
PRICE_REL_TOL = 0.01
MAX_CANDIDATES_PER_HTML = 5000
EXCLUDE_DERIVED_ARTIFACT_HTML = True

# Pages that remain completely unannotated are excluded from metric computation,
# while extraction predictions are still saved.
FAIL_ON_INCOMPLETE_GT = False

# The current industrial GT was reviewed by the user, although most rows still
# carry in_progress. "labeled" accepts those rows; "done" requires review status.
# Explicit pending/todo/new/skip rows never become labels merely by having values.
ANNOTATION_POLICY = "labeled"  # "labeled" or "done"
BENCHMARK_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]

# Optional parser-contract file. It is NOT required for the benchmark.
# If a copy is placed next to this notebook, a parity check is performed.
PARSER_CONTRACT_FILE: Path | None = None

print("BENCHMARK_ROOT =", BENCHMARK_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)
for name, cfg in DATASETS.items():
    print(f"{name}: HTML={cfg['html_dir']} reference={cfg['reference_file'] or 'AUTO'}")


BENCHMARK_ROOT = D:\Pet_Codes\science\article-price-tracker
OUTPUT_DIR = D:\Pet_Codes\science\article-price-tracker\baseline_results\determined
synthetic: HTML=D:\Pet_Codes\science\article-price-tracker\synthetic\html reference=AUTO
robustness: HTML=D:\Pet_Codes\science\article-price-tracker\synthetic\html reference=D:\Pet_Codes\science\article-price-tracker\synthetic\robustness_ground_truth.csv
industrial: HTML=D:\Pet_Codes\science\article-price-tracker\industrial\html reference=AUTO


## 2. Shared evaluation contract

Name matching uses SequenceMatcher >= 0.85; prices use ±1% tolerance.
URL identity additionally treats www and the bare host as equivalent. Query,
fragment and trailing slash are removed; other subdomains remain distinct.
The optional parser parity check compares URLs under this application policy.
HTML identity preserves the relative path, including subdirectories.


In [4]:

SRC_URL_RE = re.compile(r"^https?://(?:www\.)?([^/]+)")
JSONLD_RE = re.compile(
    r'<script[^>]*type\s*=\s*["\']application/ld\+json["\'][^>]*>(.*?)</script>',
    re.DOTALL | re.IGNORECASE,
)
BENCHMARK_FILENAME_RE = re.compile(
    r"^(?P<source>.+)-(?P<run>run-(?:[0-9a-f]+|none\d+))(?:-(?P<copy>\d+))?$",
    re.IGNORECASE,
)


def _clean_text(value: object) -> str:
    if value is None or value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return ""
    text = html_std.unescape(str(value))
    text = text.replace("\u00a0", " ").replace("\u202f", " ").replace("\u2009", " ")
    return re.sub(r"\s+", " ", text).strip()


def _is_blank(x: object) -> bool:
    if x is None:
        return True
    if pd.isna(x):
        return True
    return str(x).strip() == ""


def _first_nonempty(*values: object) -> str:
    for value in values:
        text = _clean_text(value)
        if text:
            return text
    return ""


def normalize_source(s: str | None) -> str | None:
    # Exact contract from parse_logs_to_excel_v2.py.
    if _is_blank(s):
        return None
    s = str(s).strip().strip("'").strip('"').lower()
    m = SRC_URL_RE.match(s)
    if m:
        s = m.group(1)
    if s.startswith("www."):
        s = s[4:]
    return s


def _norm_url(u: object) -> str | None:
    # Application identity: www and bare host are the same product domain.
    # Keep the legacy case/query policy; do not rewrite other subdomains.
    if _is_blank(u):
        return None
    u = str(u).strip().split("?")[0].split("#")[0].rstrip("/").lower()
    try:
        parts = urlsplit(u)
    except ValueError:
        return None
    if parts.netloc.startswith("www."):
        u = parts._replace(netloc=parts.netloc[4:]).geturl()
    return u or None


def _resolve_norm_url(value: object, base_url: str | None = None) -> str | None:
    if _is_blank(value):
        return None
    return _norm_url(urljoin(base_url or "", str(value).strip()))


def _name_match(a: str | None, b: str | None, threshold: float = 0.85) -> bool:
    if not a or not b:
        return False
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio() >= threshold


def _price_match(p1: object, p2: object, tol: float = 0.01) -> bool:
    if p1 is None or p2 is None or p1 == "" or p2 == "":
        return False
    try:
        a, b = float(p1), float(p2)
    except (ValueError, TypeError):
        return False
    if b == 0:
        return abs(a) < 1e-9
    return abs(a - b) / abs(b) <= tol


def parse_price(value: object) -> float | None:
    if value is None:
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        try:
            v = float(value)
            return v if math.isfinite(v) and v >= 0 else None
        except (TypeError, ValueError):
            return None
    text = _clean_text(value)
    if not text:
        return None
    text = re.sub(r"(?i)(?:₽|руб(?:\.|лей|ля)?|р\.)", "", text)
    text = text.replace("\u00a0", "").replace("\u202f", "").replace("\u2009", "").replace(" ", "")
    m = re.search(r"-?\d[\d.,]*", text)
    if not m:
        return None
    raw = m.group(0)
    if "," in raw and "." in raw:
        if raw.rfind(",") > raw.rfind("."):
            raw = raw.replace(".", "").replace(",", ".")
        else:
            raw = raw.replace(",", "")
    elif "," in raw:
        parts = raw.split(",")
        raw = "".join(parts) if len(parts[-1]) == 3 else ".".join(parts)
    elif raw.count(".") > 1:
        raw = raw.replace(".", "")
    try:
        v = float(raw)
    except ValueError:
        return None
    return v if math.isfinite(v) and v >= 0 else None


def _parse_binary_annotation(x: object) -> bool | None:
    if _is_blank(x):
        return None
    if isinstance(x, bool):
        return x
    try:
        return bool(int(float(x)))
    except (TypeError, ValueError):
        s = str(x).strip().casefold()
        if s in {"1", "true", "yes", "y", "да"}:
            return True
        if s in {"0", "false", "no", "n", "нет"}:
            return False
    return None


def normalize_html_key(value: object) -> str:
    if _is_blank(value):
        return ""
    value = str(value).replace("\\", "/").rstrip("/")
    value = value.removeprefix("./").casefold()
    return value.removeprefix("html/")


def parse_benchmark_filename(path_or_name: object) -> dict[str, str | None]:
    name = str(path_or_name).replace("\\", "/").split("/")[-1]
    stem = Path(name).stem
    m = BENCHMARK_FILENAME_RE.match(stem)
    if not m:
        return {"source": None, "run_id": None}
    return {
        "source": normalize_source(m.group("source")),
        "run_id": m.group("run").lower(),
    }


def host_from_url(url: object) -> str | None:
    if _is_blank(url):
        return None
    try:
        host = urlsplit(str(url)).hostname
    except Exception:
        return None
    return normalize_source(host)


def detect_derived_collection_summary(html_text: str) -> bool:
    head = html_text[:200_000].casefold()
    return (
        "collection artifact" in head
        and "<th>product</th>" in head
        and "<th>price</th>" in head
        and "<th>url</th>" in head
    )


def _coerce_cell(raw: str):
    if raw == "":
        return ""
    try:
        if re.fullmatch(r"-?\d+", raw):
            return int(raw)
        if re.fullmatch(r"-?(?:\d+\.\d*|\d*\.\d+)(?:[eE][+-]?\d+)?", raw):
            return float(raw)
    except ValueError:
        pass
    return raw


def _column_index(cell_ref: str) -> int:
    letters = re.match(r"[A-Z]+", cell_ref or "")
    if not letters:
        return 0
    n = 0
    for ch in letters.group(0):
        n = n * 26 + (ord(ch) - ord("A") + 1)
    return n - 1


def _read_xlsx_sheet_without_openpyxl(path: Path, wanted_sheet: str) -> pd.DataFrame:
    ns = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    rel_ns = {"r": "http://schemas.openxmlformats.org/package/2006/relationships"}
    office_rel = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

    with zipfile.ZipFile(path) as z:
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rel_map = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels.findall("r:Relationship", rel_ns)}
        sheet_path = None
        available = []
        for sheet in workbook.findall("x:sheets/x:sheet", ns):
            name = sheet.attrib.get("name", "")
            available.append(name)
            if name.casefold() == wanted_sheet.casefold():
                rid = sheet.attrib.get(f"{{{office_rel}}}id")
                target = rel_map.get(rid or "")
                if target:
                    target = target.lstrip("/")
                    sheet_path = target if target.startswith("xl/") else "xl/" + target
                break
        if sheet_path is None:
            raise ValueError(f"Sheet {wanted_sheet!r} not found; available={available}")

        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("x:si", ns):
                shared.append("".join((n.text or "") for n in si.findall(".//x:t", ns)))

        root = ET.fromstring(z.read(sheet_path))
        rows = []
        for row in root.findall(".//x:sheetData/x:row", ns):
            cells = {}
            max_idx = -1
            for cell in row.findall("x:c", ns):
                idx = _column_index(cell.attrib.get("r", ""))
                max_idx = max(max_idx, idx)
                ctype = cell.attrib.get("t")
                inline = cell.findall("x:is//x:t", ns)
                vnode = cell.find("x:v", ns)
                if inline:
                    value = "".join((n.text or "") for n in inline)
                else:
                    raw = vnode.text if vnode is not None and vnode.text is not None else ""
                    if ctype == "s":
                        try:
                            value = shared[int(raw)]
                        except (ValueError, IndexError):
                            value = raw
                    elif ctype == "b":
                        value = raw == "1"
                    else:
                        value = _coerce_cell(raw)
                cells[idx] = value
            if max_idx >= 0:
                rows.append([cells.get(i, "") for i in range(max_idx + 1)])

    if not rows:
        return pd.DataFrame()
    width = max(len(r) for r in rows)
    rows = [r + [""] * (width - len(r)) for r in rows]
    headers = [str(x).strip() for x in rows[0]]
    seen = {}
    clean_headers = []
    for i, h in enumerate(headers):
        h = h or f"column_{i+1}"
        count = seen.get(h, 0)
        seen[h] = count + 1
        clean_headers.append(h if count == 0 else f"{h}_{count+1}")
    return pd.DataFrame(rows[1:], columns=clean_headers)


def _read_reference_sheet(path: Path, sheet_name: str = "GroundTruth") -> pd.DataFrame:
    if path.suffix.casefold() == ".csv":
        return pd.read_csv(path)
    if path.suffix.casefold() != ".xlsx":
        raise ValueError(f"Unsupported reference format: {path.suffix}")
    try:
        with pd.ExcelFile(path) as xls:
            actual = next((s for s in xls.sheet_names if s.casefold() == sheet_name.casefold()), None)
            if actual is None:
                raise ValueError(f"Sheet {sheet_name!r} not found; available={xls.sheet_names}")
            return pd.read_excel(xls, sheet_name=actual)
    except ImportError:
        return _read_xlsx_sheet_without_openpyxl(path, sheet_name)


def discover_dataset_reference(dataset_root: Path) -> Path | None:
    preferred = (
        "dind.xlsx",
        "groundtruth.xlsx",
        "reference.xlsx",
        "synthetic_ground_truth.xlsx",
        "industrial_ground_truth.xlsx",
        "synthetic_ground_truth.csv",
        "industrial_ground_truth.csv",
        "groundtruth.csv",
    )
    for name in preferred:
        p = dataset_root / name
        if p.exists():
            return p
    candidates = []
    for pattern in ("*.xlsx", "*.csv"):
        for p in dataset_root.glob(pattern):
            if p.name.startswith("~$"):
                continue
            if re.search(r"ground|truth|dind|reference|annotation", p.name, re.I):
                candidates.append(p)
    return sorted(candidates, key=lambda p: (p.suffix != ".xlsx", p.name.casefold()))[0] if candidates else None


def discover_parser_contract_file(root: Path) -> Path | None:
    for name in ("parse_logs_to_excel_v2.py", "parse_logs_to_excel_v3.py"):
        p = root / name
        if p.exists():
            return p
    for p in root.rglob("parse_logs_to_excel_v2.py"):
        if ".venv" not in p.parts and "node_modules" not in p.parts:
            return p
    return None


def validate_parser_contract(path: Path | None) -> dict:
    if path is None or not Path(path).exists():
        return {"available": False, "path": str(path or ""), "ok": None, "mismatches": []}
    spec = importlib.util.spec_from_file_location("_baseline_reference_contract", str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)

    mismatches = []
    source_cases = [None, "bigam.ru", "WWW.DNS-SHOP.RU", "https://www.mvideo.ru/search"]
    url_cases = [None, "https://www.bigam.ru/product/123/", "https://demo.ru/p/1/?utm=x#frag"]
    name_cases = [("Дрель Alpha 500", "дрель alpha 500"), ("A-B", "A B"), (None, "x")]
    price_cases = [(10050, 10000), (10101, 10000), (0, 0), (None, 1)]

    for x in source_cases:
        if normalize_source(x) != module.normalize_source(x):
            mismatches.append(("normalize_source", x, normalize_source(x), module.normalize_source(x)))
    for x in url_cases:
        # URL identity deliberately extends the parser contract with www folding.
        if _norm_url(x) != _norm_url(module._norm_url(x)):
            mismatches.append(("_norm_url", x, _norm_url(x), module._norm_url(x)))
    for a, b in name_cases:
        if _name_match(a, b, NAME_THRESHOLD) != module._name_match(a, b, NAME_THRESHOLD):
            mismatches.append(("_name_match", (a, b), _name_match(a, b), module._name_match(a, b)))
    for a, b in price_cases:
        if _price_match(a, b, PRICE_REL_TOL) != module._price_match(a, b, PRICE_REL_TOL):
            mismatches.append(("_price_match", (a, b), _price_match(a, b), module._price_match(a, b)))
    return {"available": True, "path": str(path), "ok": not mismatches,
            "url_policy": "legacy rules plus www equivalence and trailing-slash normalization",
            "mismatches": mismatches}


if PARSER_CONTRACT_FILE is None:
    PARSER_CONTRACT_FILE = discover_parser_contract_file(BENCHMARK_ROOT)

contract_check = validate_parser_contract(PARSER_CONTRACT_FILE)
print("Parser contract:", json.dumps(contract_check, ensure_ascii=False, indent=2))


Parser contract: {
  "available": true,
  "path": "D:\\Pet_Codes\\science\\article-price-tracker\\scripts\\parse_logs_to_excel_v2.py",
  "ok": true,
  "url_policy": "legacy rules plus www equivalence and trailing-slash normalization",
  "mismatches": []
}


## 3. Dataset manifests and GroundTruth

When the reference workbook has a Pages sheet, it defines the corpus. Extra
HTML files are ignored; missing files remain in the manifest with missing_html.
Without Pages, HTML is discovered recursively. Relative paths remain unique.

GroundTruth accepts html_file, page_id or synthetic mutation_id. Original
entity_id/gt_id and additional classification columns are retained. Missing
identifiers in legacy synthetic references receive deterministic fallback IDs.

ANNOTATION_POLICY='labeled' includes the current user-reviewed in_progress
annotations. Use 'done' to require done/reviewed/complete/completed statuses.
Explicit pending/todo/new/unannotated rows are never scored. A partially
annotated page is excluded as a whole; skip/excluded excludes its whole page.

Industrial price 0 means absent. A blank price is absent only with an explicit
price state/confirmation or the previously recorded USER_REVIEW: no price note.
Otherwise its state is unannotated. Synthetic zero remains a numeric price.


In [5]:
def _canonical_columns(raw: pd.DataFrame) -> dict[str, str]:
    return {re.sub(r"[^a-z0-9]+", "", str(c).casefold()): c for c in raw.columns}


def _pick_column(raw: pd.DataFrame, aliases: list[str]) -> pd.Series | None:
    canonical = _canonical_columns(raw)
    for alias in aliases:
        key = re.sub(r"[^a-z0-9]+", "", alias.casefold())
        if key in canonical:
            return raw[canonical[key]]
    return None


def _identity(prefix: str, *parts: object) -> str:
    value = "\x1f".join("" if _is_blank(p) else str(p) for p in parts)
    return prefix + hashlib.sha256(value.encode("utf-8")).hexdigest()[:20]


def read_groundtruth(path: Path, sheet_name: str = "GroundTruth", *,
                     zero_price_is_missing: bool = False,
                     annotation_policy: str = ANNOTATION_POLICY) -> pd.DataFrame:
    if annotation_policy not in {"labeled", "done"}:
        raise ValueError(f"Unknown annotation policy: {annotation_policy}")
    raw = _read_reference_sheet(path, sheet_name)

    def col(aliases, default=None):
        result = _pick_column(raw, aliases)
        return result if result is not None else pd.Series([default] * len(raw), index=raw.index, dtype=object)

    # Keep classification/cluster columns and the original annotations for joins.
    gt = raw.copy()
    for name, aliases in {
        "gt_id": ["gt_id"], "page_id": ["page_id", "mutation_id", "base_page_id"],
        "html_file": ["html_file", "benchmark_html_file", "file", "filename"],
        "entity_id": ["entity_id"], "source": ["source", "source_name", "supplier", "site"],
        "run_id": ["run_id", "run", "collection_run_id"], "query": ["query"],
        "annotation_status": ["annotation_status", "status"], "is_product": ["is_product"],
        "gt_name": ["name_gt", "gt_name", "ground_truth_name", "product_name_gt", "product_name"],
        "gt_price": ["price_gt", "gt_price", "ground_truth_price", "product_price"],
        "product_url": ["product_url_gt", "product_url", "canonical_url", "url"],
        "notes": ["notes"],
    }.items():
        gt[name] = col(aliases)
    missing_html = gt.html_file.map(_is_blank)
    has_page = ~gt.page_id.map(_is_blank)
    gt.loc[missing_html & has_page, "html_file"] = gt.loc[missing_html & has_page, "page_id"].astype(str) + ".html"
    gt["html_key"] = gt.html_file.map(normalize_html_key)
    parsed = gt.html_file.map(parse_benchmark_filename)
    gt["source"] = [normalize_source(src) or p["source"] for src, p in zip(gt.source, parsed)]
    gt["run_id"] = [("" if _is_blank(r) else str(r).strip()) or p["run_id"] or "" for r, p in zip(gt.run_id, parsed)]
    gt["gt_name"] = gt.gt_name.map(lambda x: "" if _is_blank(x) else str(x).strip())
    price = pd.to_numeric(gt.gt_price, errors="coerce")
    price = price.where(price.map(lambda p: pd.notna(p) and math.isfinite(p) and p >= 0))
    zero_absent = price.eq(0) & zero_price_is_missing
    gt["gt_price"] = price.mask(zero_absent)
    gt["product_url"] = gt.product_url.map(_norm_url)
    gt["is_product"] = gt.is_product.map(_parse_binary_annotation)
    gt["annotation_status"] = gt.annotation_status.map(lambda x: "" if _is_blank(x) else str(x).strip().casefold())
    gt["_excluded"] = gt.annotation_status.isin({"skip", "excluded"})
    reviewed = {"done", "reviewed", "complete", "completed"}
    pending = {"pending", "todo", "unannotated", "new"}

    def annotated(row):
        status = row.annotation_status
        if row._excluded or status in pending:
            return False
        if annotation_policy == "done":
            return status in reviewed
        return bool(status in reviewed | {"in_progress", "annotated"}
                    or (not status and (row.is_product is not None or row.gt_name
                                        or pd.notna(row.gt_price) or row.product_url)))

    gt["_annotated"] = pd.Series([annotated(r) for _, r in gt.iterrows()], index=gt.index, dtype=bool)
    gt["_positive"] = pd.Series([
        bool(r._annotated and (r.is_product is True or (r.is_product is not False and
             (r.gt_name or pd.notna(r.gt_price) or r.product_url))))
        for _, r in gt.iterrows()
    ], index=gt.index, dtype=bool)

    explicit_state = col(["gt_price_state", "price_state_gt", "price_state"])
    confirmed = col(["price_absence_confirmed"]).map(_parse_binary_annotation).eq(True)
    # These notes were written by the prior, explicitly user-confirmed GT update.
    confirmed |= gt.notes.fillna("").astype(str).str.contains("USER_REVIEW: no price", regex=False)
    states = []
    for i in gt.index:
        state = "" if _is_blank(explicit_state.loc[i]) else str(explicit_state.loc[i]).strip().casefold()
        inferred = "present" if pd.notna(gt.at[i, "gt_price"]) else (
            "absent" if zero_absent.loc[i] or confirmed.loc[i] else "unannotated")
        state = state or inferred
        if state not in {"present", "absent", "unannotated"}:
            raise ValueError(f"Unknown price state at GT row {i}: {state}")
        if (state == "present") != pd.notna(gt.at[i, "gt_price"]):
            raise ValueError(f"Price/state conflict at GT row {i}: {state}")
        states.append(state)
    gt["gt_price_state"] = pd.Series(states, index=gt.index, dtype=object)
    gt["gt_id"] = gt.gt_id.astype(object)
    gt["entity_id"] = gt.entity_id.astype(object)
    gt["_eval_product_key"] = gt.product_url.astype(object)
    for i in gt.index:
        if gt.at[i, "_positive"] and _is_blank(gt.at[i, "entity_id"]):
            gt.at[i, "entity_id"] = _identity("entity-", gt.at[i, "html_key"], gt.at[i, "product_url"] or f"row-{i}")
        if _is_blank(gt.at[i, "gt_id"]):
            entity = gt.at[i, "entity_id"]
            gt.at[i, "gt_id"] = _identity("gt-", gt.at[i, "html_key"], entity if not _is_blank(entity) else f"row-{i}")
        if gt.at[i, "_positive"] and _is_blank(gt.at[i, "product_url"]):
            gt.at[i, "_eval_product_key"] = "__gt_missing_url__:" + str(gt.at[i, "entity_id"])
    gt.attrs.update(source_path=str(path), reference_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                    annotation_policy=annotation_policy, zero_price_is_missing=zero_price_is_missing)
    return gt.reset_index(drop=True)


def read_pages_metadata(reference_file: Path | None) -> dict[str, dict] | None:
    if reference_file is None:
        return None
    if reference_file.suffix.casefold() == ".csv":
        # Generated corpora have an authoritative companion *_pages.csv.
        stem = reference_file.stem
        if not stem.endswith("_ground_truth"):
            return None
        pages_file = reference_file.with_name(stem.removesuffix("_ground_truth") + "_pages.csv")
        if not pages_file.is_file():
            return None
        pages = pd.read_csv(pages_file)
    elif reference_file.suffix.casefold() == ".xlsx":
        try:
            pages = _read_reference_sheet(reference_file, "Pages")
        except ValueError as exc:
            if "not found; available=" in str(exc):
                return None
            raise
    else:
        return None
    file_col = _pick_column(pages, ["benchmark_html_file", "html_file"])
    if file_col is None:
        raise ValueError("Pages sheet is missing benchmark_html_file/html_file")
    mapping = {}
    for i, row in pages.iterrows():
        key = normalize_html_key(file_col.loc[i])
        if not key or key in mapping:
            raise ValueError(f"Empty or duplicate Pages HTML key: {key!r}")
        mapping[key] = row.to_dict()
    return mapping


def build_html_manifest(dataset_name: str, html_dir: Path, reference_file: Path | None = None) -> pd.DataFrame:
    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "html_path",
               "page_url", "source_url", "query", "location", "route", "artifact_kind", "skip_reason",
               "content_sha256", "expected_sha256"]
    metadata = read_pages_metadata(reference_file)
    if metadata is not None:
        paths = [(html_dir / key, key, meta) for key, meta in metadata.items()]
    else:
        paths = [(p, normalize_html_key(p.relative_to(html_dir)), {})
                 for p in sorted([*html_dir.rglob("*.html"), *html_dir.rglob("*.htm")])]
    rows = []
    for html_path, html_key, meta in paths:
        if not html_path.resolve().is_relative_to(html_dir.resolve()):
            raise ValueError(f"HTML path outside dataset: {html_path}")
        parsed = parse_benchmark_filename(html_path.name)
        page_url = _first_nonempty(meta.get("page_url"), meta.get("source_url"))
        reason = ""
        if _parse_binary_annotation(meta.get("include")) is False:
            reason = _first_nonempty(meta.get("exclusion_reason"), "excluded_in_pages")
        exists = html_path.is_file()
        data = html_path.read_bytes() if exists else b""
        text = data.decode("utf-8", errors="replace")
        if not exists:
            reason = "missing_html"
        if not page_url and exists:
            page_url = document_canonical_url(BeautifulSoup(text, "lxml"), "")
        derived = detect_derived_collection_summary(text)
        if not reason and derived and EXCLUDE_DERIVED_ARTIFACT_HTML:
            reason = "derived_from_pipeline_snapshots"
        rows.append({
            "dataset": dataset_name, "page_id": _first_nonempty(meta.get("page_id"), html_key.removesuffix(html_path.suffix)),
            "html_file": "html/" + html_key, "html_key": html_key, "html_path": str(html_path),
            "run_id": _first_nonempty(meta.get("run_id"), parsed["run_id"]),
            "source": normalize_source(meta.get("source")) or parsed["source"] or host_from_url(page_url) or "unknown",
            "page_url": page_url, "source_url": _first_nonempty(meta.get("source_url"), page_url),
            "query": _first_nonempty(meta.get("query")), "location": _first_nonempty(meta.get("location")),
            "route": _first_nonempty(meta.get("route")),
            "artifact_kind": "derived_collection_summary" if derived else "raw_html", "skip_reason": reason,
            "content_sha256": hashlib.sha256(data).hexdigest() if exists else "",
            "expected_sha256": _first_nonempty(meta.get("sha256")),
        })
        rows[-1].update({"page_meta__" + str(k): v for k, v in meta.items()})
    counters = {}
    for row in rows:
        if not row["run_id"]:
            source = row["source"]
            counters[source] = counters.get(source, 0) + 1
            row["run_id"] = f"run-none{counters[source]}"
    extra_columns = sorted({k for r in rows for k in r if k not in columns})
    manifest = pd.DataFrame(rows, columns=columns + extra_columns)
    if manifest.html_key.duplicated().any() or manifest.page_id.duplicated().any():
        raise ValueError("Manifest requires unique html_key and page_id")
    manifest.attrs["selection"] = "Pages" if metadata is not None else "html_directory"
    return manifest


def align_groundtruth_to_manifest(gt: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    out = gt.copy()
    if gt.empty or manifest.empty:
        return out
    if manifest.html_key.duplicated().any() or manifest.page_id.duplicated().any():
        raise ValueError("Duplicate HTML/page keys in manifest")
    meta = manifest.set_index("html_key")
    for i in out.index:
        key = out.at[i, "html_key"]
        if key not in meta.index:
            continue
        for col in ["html_file", "page_id", "run_id", "source", "query"]:
            if col in meta and _is_blank(out.at[i, col]):
                out.at[i, col] = meta.at[key, col]
        if str(out.at[i, "page_id"]) != str(meta.at[key, "page_id"]):
            raise ValueError(f"GT/manifest page_id conflict: {key}")
    return out


def evaluation_scope(gt: pd.DataFrame, manifest: pd.DataFrame):
    """Only available, fully annotated pages; skip excludes the whole page."""
    if manifest.html_key.duplicated().any():
        raise ValueError("Duplicate HTML keys in manifest")
    excluded = set(gt.loc[gt["_excluded"], "html_key"])
    active = manifest[manifest.skip_reason.fillna("").eq("") & ~manifest.html_key.isin(excluded)].copy()
    local_gt = gt[gt.html_key.isin(active.html_key)].copy()
    complete = local_gt.groupby("html_key")["_annotated"].all()
    keys = set(complete.index[complete])
    return local_gt[local_gt.html_key.isin(keys)].copy(), active[active.html_key.isin(keys)].copy()


def groundtruth_audit(gt: pd.DataFrame, manifest: pd.DataFrame) -> dict:
    scoped_gt, scored = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive]
    excluded = set(gt.loc[gt._excluded, "html_key"])
    active_keys = set(manifest.loc[manifest.skip_reason.fillna("").eq(""), "html_key"]) - excluded
    duplicate_keys = int(positive[positive.product_url.notna()].duplicated(["html_key", "product_url"], keep=False).sum())
    missing_name = sum(_is_blank(v) for v in positive.gt_name)
    missing_url = int(positive.product_url.isna().sum())
    unknown_price = int(positive.gt_price_state.eq("unannotated").sum())
    missing_files = manifest.loc[manifest.skip_reason.eq("missing_html"), "html_file"].tolist()
    hash_mismatch = []
    if {"content_sha256", "expected_sha256"} <= set(manifest.columns):
        hash_mismatch = manifest.loc[
            manifest.content_sha256.fillna("").ne("") & manifest.expected_sha256.fillna("").ne("")
            & manifest.content_sha256.ne(manifest.expected_sha256), "html_file"].tolist()
    ready = bool(active_keys and active_keys <= set(scored.html_key) and not
                 (missing_name or missing_url or unknown_price or duplicate_keys or hash_mismatch))
    outside = set(gt.loc[gt._annotated, "html_key"]) - set(manifest.html_key)
    return {
        "reference_rows": len(gt), "manifest_pages": len(manifest), "available_manifest_pages": len(active_keys),
        "annotated_pages": len(scored), "unannotated_manifest_pages": len(active_keys - set(scored.html_key)),
        "reference_pages_not_in_html_dir": len(outside), "positive_entities": len(positive),
        "negative_or_empty_rows": int((scoped_gt._annotated & ~scoped_gt._positive).sum()),
        "positive_missing_name_gt": missing_name, "positive_missing_price_gt": int(positive.gt_price.isna().sum()),
        "positive_confirmed_absent_price_gt": int(positive.gt_price_state.eq("absent").sum()),
        "positive_unannotated_price_gt": unknown_price, "positive_missing_product_url_gt": missing_url,
        "duplicate_positive_page_url_keys": duplicate_keys, "missing_html_files": missing_files,
        "html_hash_mismatches": hash_mismatch, "excluded_gt_pages": len(excluded),
        "annotation_policy": gt.attrs.get("annotation_policy", ANNOTATION_POLICY),
        "annotation_status_counts": gt.annotation_status.value_counts().to_dict(),
        "ready_for_available_pages": ready,
        "paper_ready_closed_world": ready and not missing_files and not outside,
    }


## 4. Embedded original parser and historical controls

The following cell embeds seven unmodified original TypeScript modules, their SHA256 checksums,
and a standalone JavaScript bundle with the exact dependency versions used by the reference run.
`prepare_original_runtime()` verifies and materializes these files in `.benchmark_runtime/`.
This folder is disposable and is recreated from notebook contents. No original repository is read.

The original parser includes its DOM/CSS, JSON-LD, embedded-state JSON, supplier profiles,
candidate selection, field cleaning and query relevance logic. The request limit remains 500.
The Python B1/B2/B3 functions are historical controls, activated only if explicitly added to VARIANTS.


In [6]:
# Frozen original PriceTracker parser sources shared by deterministic and hybrid notebooks.
ORIGINAL_SOURCE_FILES_PATH = BENCHMARK_ROOT / "config" / "original_fast_renderer" / "source_files.json"
if not ORIGINAL_SOURCE_FILES_PATH.is_file():
    raise FileNotFoundError(f"Missing frozen parser sources: {ORIGINAL_SOURCE_FILES_PATH}")
ORIGINAL_SOURCE_FILES = json.loads(ORIGINAL_SOURCE_FILES_PATH.read_text(encoding="utf-8"))



# Compiled original sources + exact installed npm dependencies; no runtime npm lookup.



# Optional historical baselines (not selected by default).
IN_STOCK_FALSE = ("outofstock", "out of stock", "unavailable", "sold out", "нет в наличии", "отсутств", "законч")
IN_STOCK_TRUE = ("instock", "in stock", "в наличии", "available", "есть", "доступ")


# ---- B1 core copied from parse_logs_to_excel_v2.py ----
def _iter_jsonld_nodes_reference(data):
    if isinstance(data, list):
        for x in data:
            yield from _iter_jsonld_nodes_reference(x)
    elif isinstance(data, dict):
        if "@graph" in data:
            yield from _iter_jsonld_nodes_reference(data["@graph"])
        yield data


def _node_to_product_reference(node: dict) -> dict:
    offers = node.get("offers") or node.get("Offer") or {}
    if isinstance(offers, list):
        offers = offers[0] if offers else {}
    price = None
    availability = None
    for src in (offers, node):
        if isinstance(src, dict):
            p = src.get("price") or src.get("lowPrice")
            if p is not None and price is None:
                try:
                    price = float(str(p).replace(",", ".").replace(" ", ""))
                except (ValueError, TypeError):
                    pass
            a = src.get("availability")
            if a and availability is None:
                availability = str(a)
    return {
        "name": node.get("name"),
        "url": node.get("url") or (offers.get("url") if isinstance(offers, dict) else None),
        "price": price,
        "availability": availability,
    }


def extract_jsonld_products_reference(html_text: str) -> list[dict]:
    products: list[dict] = []
    for m in JSONLD_RE.finditer(html_text):
        raw = m.group(1).strip()
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            try:
                data = json.loads(re.sub(r",\s*([}\]])", r"\1", raw))
            except json.JSONDecodeError:
                continue
        for node in _iter_jsonld_nodes_reference(data):
            if not isinstance(node, dict):
                continue
            t = node.get("@type")
            types = [t] if isinstance(t, str) else (t or [])
            if not any("product" in str(x).lower() for x in types):
                continue
            products.append(_node_to_product_reference(node))
    return products

def parse_in_stock(value: object) -> bool | None:
    text = _clean_text(value).casefold().replace("https://schema.org/", "").replace("http://schema.org/", "")
    if not text:
        return None
    if any(x in text for x in IN_STOCK_FALSE):
        return False
    if any(x in text for x in IN_STOCK_TRUE):
        return True
    if text in {"0", "false", "no", "нет"}:
        return False
    if text in {"1", "true", "yes", "да"}:
        return True
    return None


def first_json_text(value: object) -> str:
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, list):
        for item in value:
            text = first_json_text(item)
            if text:
                return text
    if isinstance(value, dict):
        for key in ("@id", "url", "name", "text", "value"):
            text = first_json_text(value.get(key))
            if text:
                return text
    return ""


def is_product_type(raw_type: object) -> bool:
    values = raw_type if isinstance(raw_type, list) else [raw_type]
    for value in values:
        text = _clean_text(value).casefold().rstrip("/#")
        if text == "product" or text.endswith("/product") or text.endswith(":product"):
            return True
    return False


def walk_jsonld_products(value: object):
    if isinstance(value, list):
        for item in value:
            yield from walk_jsonld_products(item)
        return
    if not isinstance(value, dict):
        return
    if is_product_type(value.get("@type")):
        yield value
    for nested in value.values():
        if isinstance(nested, (dict, list)):
            yield from walk_jsonld_products(nested)


def select_priced_offer(value: object) -> dict | None:
    if isinstance(value, dict):
        if any(k in value for k in ("price", "lowPrice", "highPrice")):
            return value
        for key in ("offers", "priceSpecification"):
            found = select_priced_offer(value.get(key))
            if found is not None:
                return found
    elif isinstance(value, list):
        for item in value:
            found = select_priced_offer(item)
            if found is not None:
                return found
    return None


def document_base_url(soup: BeautifulSoup, fallback: str = "") -> str:
    """Choose a deterministic base URL for resolving relative product links.

    Prefer an explicit <base>; then a page canonical/og:url; finally the debug-log
    page/source URL. This compensates for older artifacts whose debug log did not
    persist the final search-page URL.
    """
    base = soup.select_one("base[href]")
    if base and _clean_text(base.get("href")):
        return urljoin(fallback, _clean_text(base.get("href")))
    for selector, attr in (
        ('link[rel="canonical"][href]', "href"),
        ('meta[property="og:url"][content]', "content"),
    ):
        node = soup.select_one(selector)
        if node and _clean_text(node.get(attr)):
            return urljoin(fallback, _clean_text(node.get(attr)))
    return fallback


def document_canonical_url(soup: BeautifulSoup, fallback: str = "") -> str:
    for selector, attr in (
        ('link[rel="canonical"][href]', "href"),
        ('meta[property="og:url"][content]', "content"),
    ):
        node = soup.select_one(selector)
        if node:
            value = _clean_text(node.get(attr))
            if value:
                return _resolve_norm_url(value, fallback)
    return _norm_url(fallback)


def extract_b1_jsonld(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    """B1: reference JSON-LD parser + only URL resolution/output adaptation."""
    base_url = document_base_url(soup, page_url)
    products = extract_jsonld_products_reference(html_text)
    canonical = document_canonical_url(soup, page_url)
    rows = []
    for p in products:
        name = _clean_text(p.get("name")) or None
        price = p.get("price")
        raw_url = _clean_text(p.get("url"))
        product_url = _norm_url(urljoin(base_url, raw_url)) if raw_url else None
        if not product_url and len(products) == 1:
            product_url = canonical
        if not product_url:
            continue
        if not name and price is None:
            continue
        rows.append({
            "product_url": product_url,
            "pred_name": name,
            "pred_price": price,
            "pred_in_stock": parse_in_stock(p.get("availability")),
            "extract_notes": "jsonld_product_reference_core",
        })
    return rows


B2_CARD_SELECTORS = (
    # schema/data attributes
    '[itemtype*="Product"]',
    '[itemscope][itemtype*="schema.org/Product"]',
    '[data-product-id]', '[data-meta-product-id]', '[data-product]', '[data-offer-id]',
    '[data-sku]', '[data-id="product"]', '[data-entity="item"]',
    '[data-testid*="product" i]', '[data-test*="product" i]', '[data-qa*="product" i]',
    # repeated generic ecommerce cards
    '.catalog__list-item.snippet', '.promo-slider__item.snippet', '.snippet', '.snippet__content',
    '[class~="snippet"]', '[class*="snippet" i]',
    '.product-card-list', '.product-card', '[class*="product-card" i]', '[class*="product_card" i]',
    '.catalog-product', '[class*="catalog-product" i]', '.catalog-item', '.catalog_item',
    '.product-item', '.product_item', '.goods-item', '.goods_item', '.item-card', '.card-item',
    '[class*="tile" i]',
    # broad structural containers are intentionally last
    'article', 'li', 'tr',
)

B2_NAME_SELECTORS = (
    '[itemprop="name"]',
    '.snippet__title', '.snippet-title',
    '.product-card__name', '.product-card__title',
    '[class*="product-name" i]', '[class*="product-title" i]',
    '[class*="title" i]', '[class*="name" i]',
    '[data-qa*="name" i]', '[data-testid*="name" i]',
    'h1', 'h2', 'h3', 'a[title]', 'img[alt]',
)

B2_PRICE_SELECTORS = (
    'meta[itemprop="price"]', '[content][itemprop="price"]', '[itemprop="price"]',
    '[data-price]', '[data-retail-price]', '[data-meta-price]',
    '.snippet-price__value', '.product-card__price-current', '.current-price', '.price-main',
    '[class*="price" i]:not([class*="old" i]):not([class*="credit" i]):not([class*="month" i]):not([class*="installment" i])',
    '[data-qa*="price" i]', '[data-testid*="price" i]',
)

B2_EXCLUDED_HREF_PARTS = (
    '/cart', '/basket', '/compare', '/favorites', '/favorite', '/wishlist', '/login', '/auth',
    '/search', '/filter', '/delivery', '/payment', '/help', '/contacts', '/about', '/news',
    '/blog', '/reviews', '/review',
)
B2_PRODUCT_HREF_PARTS = (
    '/product/', '/products/', '/goods/', '/good/', '/katalog/', '/shop/', '/item/', '/items/',
    '/sku/', '/p/', '/tovar/', '/product-', '/item-', '/goods-', '/offer/', '/offers/', '/books/',
)


def looks_like_product_href(value: object) -> bool:
    href = _clean_text(value).casefold()
    if not href or href == '/' or href.startswith(('#', 'javascript:', 'mailto:', 'tel:')):
        return False
    if any(part in href for part in B2_EXCLUDED_HREF_PARTS):
        return False
    if any(part in href for part in B2_PRODUCT_HREF_PARTS):
        return True
    if re.search(r'[?&](?:product|sku|item|offer|good|book)[_-]?id=', href):
        return True
    if re.search(r'/catalog/(?:product|goods|item|tovar|sku|offer)[^?#]*', href):
        return True
    if re.search(r'/catalog/[^?#]+/(?:product|goods|item|tovar|sku|offer)[^?#]*', href):
        return True
    if re.search(r'/catalog/[^?#]+/[^?#]*\d{3,}', href):
        return True
    return False


def _tag_value(node, attrs=("content", "data-price", "data-retail-price", "data-meta-price", "value")) -> str:
    for attr in attrs:
        value = node.get(attr) if hasattr(node, "get") else None
        if _clean_text(value):
            return _clean_text(value)
    return _clean_text(node.get_text(" ", strip=True)) if hasattr(node, "get_text") else ""


def _first_selector_value(root, selectors, *, attr_names=("content", "value")) -> tuple[str, str]:
    for selector in selectors:
        try:
            nodes = root.select(selector)
        except Exception:
            continue
        for node in nodes:
            value = ""
            for attr in attr_names:
                value = _clean_text(node.get(attr))
                if value:
                    break
            if not value:
                value = _clean_text(node.get_text(" ", strip=True))
            if value:
                return value, selector
    return "", ""


def _collect_unique_nodes(soup: BeautifulSoup, selectors, limit: int) -> list:
    nodes = []
    seen = set()
    for selector in selectors:
        try:
            found = soup.select(selector)
        except Exception:
            continue
        for node in found:
            ident = id(node)
            if ident in seen:
                continue
            seen.add(ident)
            nodes.append(node)
            if len(nodes) >= limit:
                return nodes
    return nodes


def _b2_candidate_nodes(soup: BeautifulSoup) -> tuple[list, str]:
    # Collect from the whole fixed selector set rather than stopping at the first
    # selector. This avoids a stray matching widget suppressing real product cards.
    nodes = _collect_unique_nodes(soup, B2_CARD_SELECTORS, MAX_CANDIDATES_PER_HTML)
    if nodes:
        return nodes, "generic_card_union"

    # Product-link fallback: climb a few levels and retain the nearest ancestor
    # that carries a price signal. Link classification is source-agnostic.
    links = [a for a in soup.find_all("a", href=True) if looks_like_product_href(a.get("href"))]
    if links:
        nodes = []
        seen = set()
        for link in links[:MAX_CANDIDATES_PER_HTML]:
            candidate = link
            current = link
            for _ in range(5):
                parent = getattr(current, "parent", None)
                if parent is None or not hasattr(parent, "select"):
                    break
                if any(parent.select_one(sel) is not None for sel in B2_PRICE_SELECTORS):
                    candidate = parent
                    break
                current = parent
            ident = id(candidate)
            if ident not in seen:
                seen.add(ident)
                nodes.append(candidate)
        if nodes:
            return nodes[:MAX_CANDIDATES_PER_HTML], "product_link_parent"

    # Product-detail fallback: one document-level candidate only when a title and a price signal exist.
    has_name = bool(soup.select_one('[itemprop="name"], h1, title'))
    has_price = any(soup.select_one(sel) is not None for sel in B2_PRICE_SELECTORS)
    if has_name and has_price and soup.body is not None:
        return [soup.body], "document_detail_fallback"
    return [], "none"


def _extract_b2_name(root) -> tuple[str, str]:
    for selector in B2_NAME_SELECTORS:
        try:
            nodes = root.select(selector)
        except Exception:
            nodes = []
        for node in nodes:
            value = _first_nonempty(
                node.get("content"), node.get("title"), node.get("aria-label"), node.get("alt"),
                node.get_text(" ", strip=True),
            )
            # Mirrors the production cleanProductName idea: discard appended widget JSON.
            value = re.split(r"\s+\{", value, maxsplit=1)[0].strip()
            value = re.sub(r"^(?:открыть карточку товара|купить|в корзину|подробнее)\s+", "", value, flags=re.I)
            if 3 <= len(value) <= 240:
                return value, selector

    # Production also falls back to the selected product-link label. Keep this
    # generic: only links that pass looks_like_product_href are considered.
    try:
        links = root.find_all("a", href=True)
    except Exception:
        links = []
    for link in links:
        if not looks_like_product_href(link.get("href")):
            continue
        value = _first_nonempty(link.get_text(" ", strip=True), link.get("title"), link.get("aria-label"))
        value = re.split(r"\s+\{", value, maxsplit=1)[0].strip()
        if 3 <= len(value) <= 240:
            return value, "product_link_text"
    return "", ""


def _extract_b2_price(root) -> tuple[float | None, str]:
    for selector in B2_PRICE_SELECTORS:
        try:
            nodes = root.select(selector)
        except Exception:
            nodes = []
        for node in nodes:
            classes = " ".join(node.get("class") or []).casefold()
            if any(bad in classes for bad in ("old", "credit", "month", "installment")):
                continue
            raw = _tag_value(node)
            if re.search(r"артикул|article|код товара", raw, flags=re.I):
                continue
            price = parse_price(raw)
            if price is not None:
                return price, selector
    return None, ""


def _extract_b2_url(root, soup: BeautifulSoup, base_url: str, card_mode: str) -> tuple[str, str]:
    # Explicit schema.org URL has highest priority.
    try:
        node = root.select_one('a[itemprop="url"][href]')
    except Exception:
        node = None
    if node and _clean_text(node.get("href")):
        return _resolve_norm_url(node.get("href"), base_url), 'a[itemprop="url"]'

    best = None
    best_score = -1
    try:
        links = root.find_all("a", href=True)
    except Exception:
        links = []
    for link in links:
        href = _clean_text(link.get("href"))
        if not looks_like_product_href(href):
            continue
        label = _first_nonempty(link.get_text(" ", strip=True), link.get("title"), link.get("aria-label"))
        cls = " ".join(link.get("class") or []).casefold()
        score = 1 + int(len(label) >= 8) * 2 + int(bool(re.search(r"title|name|product|goods|item", cls))) * 2
        if score > best_score:
            best = link
            best_score = score
    if best is not None:
        return _resolve_norm_url(best.get("href"), base_url), "best_product_link"

    if card_mode == "document_detail_fallback":
        url = document_canonical_url(soup, base_url)
        if url:
            return url, "canonical_or_page_url"
    return "", ""


def _extract_b2_stock(root) -> bool | None:
    node = root.select_one('[itemprop="availability"]') if hasattr(root, "select_one") else None
    if node:
        value = _first_nonempty(node.get("href"), node.get("content"), node.get_text(" ", strip=True))
        parsed = parse_in_stock(value)
        if parsed is not None:
            return parsed
    text = _clean_text(root.get_text(" ", strip=True)) if hasattr(root, "get_text") else ""
    return parse_in_stock(text)


def extract_b2_css(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    del html_text  # B2 intentionally uses only DOM/CSS heuristics.
    base_url = document_base_url(soup, page_url)
    nodes, card_mode = _b2_candidate_nodes(soup)
    rows = []
    for node in nodes:
        name, name_sel = _extract_b2_name(node)
        price, price_sel = _extract_b2_price(node)
        product_url, url_sel = _extract_b2_url(node, soup, base_url, card_mode)
        if not product_url:
            continue
        if not name and price is None:
            continue
        rows.append({
            "product_url": product_url,
            "pred_name": name or None,
            "pred_price": price,
            "pred_in_stock": _extract_b2_stock(node),
            "extract_notes": f"card={card_mode};name={name_sel or '-'};price={price_sel or '-'};url={url_sel or '-'}",
        })
    return rows


B3_PRICE_RE = re.compile(
    r"(?<!\d)(\d{1,3}(?:[ \u00a0\u202f\u2009]\d{3})+(?:[.,]\d{1,2})?|\d{1,9}(?:[.,]\d{1,2})?)\s*(?:₽|руб(?:\.|лей|ля)?|р\.)",
    re.I,
)


def extract_b3_regex(html_text: str, soup: BeautifulSoup, page_url: str) -> list[dict]:
    del html_text
    h1 = soup.find("h1")
    title = soup.find("title")
    name = _first_nonempty(
        h1.get_text(" ", strip=True) if h1 else "",
        title.get_text(" ", strip=True) if title else "",
    )
    visible_text = _clean_text(soup.get_text(" ", strip=True))
    m = B3_PRICE_RE.search(visible_text)
    price = parse_price(m.group(1)) if m else None
    product_url = document_canonical_url(soup, page_url)
    if not product_url or (not name and price is None):
        return []
    return [{
        "product_url": product_url,
        "pred_name": name or None,
        "pred_price": price,
        "pred_in_stock": parse_in_stock(visible_text),
        "extract_notes": "document_h1_or_title+currency_regex",
    }]


EXTRACTORS = {
    "b1_jsonld": extract_b1_jsonld,
    "b2_css": extract_b2_css,
    "b3_regex": extract_b3_regex,
}
# Frozen original parser payloads are stored as benchmark assets instead of notebook literals.
ORIGINAL_PARSER_ASSET_DIR = BENCHMARK_ROOT / "config" / "original_fast_renderer"
ORIGINAL_PARSER_META = json.loads((ORIGINAL_PARSER_ASSET_DIR / "provenance.json").read_text(encoding="utf-8"))



## 5. Run identical extraction over the selected HTML corpora

Every enabled baseline receives exactly the same saved HTML file within a dataset. No GroundTruth information is used during extraction or candidate deduplication.


In [7]:
def candidate_completeness(row: dict) -> tuple[int, int, int]:
    return (int(bool(_clean_text(row.get("pred_name")))),
            int(parse_price(row.get("pred_price")) is not None),
            int(not _is_blank(row.get("pred_in_stock"))))


def deduplicate_predictions(rows: list[dict]) -> list[dict]:
    best = {}
    for row in rows:
        key = _norm_url(row.get("product_url"))
        if not key:
            continue
        row = dict(row, product_url=key)
        if key not in best or candidate_completeness(row) > candidate_completeness(best[key]):
            best[key] = row
    return list(best.values())


EXTRACTION_LOG_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "page_id", "html_key", "html_file", "source",
    "extraction_status", "skip_reason", "html_read_ms", "html_parse_ms", "extract_ms",
    "candidate_count", "deduplicated_count", "returned_count", "candidate_limit_hit",
    "error_type", "error_message", "error_traceback",
]


def run_legacy_baselines(manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    out, log = [], []
    for rec in manifest.to_dict("records"):
        common = {k: rec.get(k) for k in ["dataset", "page_id", "html_key", "html_file", "source", "skip_reason"]}
        common.update(benchmark_run_id=BENCHMARK_RUN_ID, html_read_ms=None, html_parse_ms=None)
        if not _is_blank(rec.get("skip_reason")):
            log.extend(dict(common, variant=v, extraction_status="skipped") for v in variants)
            continue
        stage = "read_error"
        try:
            started = perf_counter()
            html_text = Path(rec["html_path"]).read_text(encoding="utf-8", errors="replace")
            common["html_read_ms"] = (perf_counter() - started) * 1000
            stage = "parse_error"
            started = perf_counter()
            soup = BeautifulSoup(html_text, "lxml")
            page_url = _first_nonempty(rec.get("page_url"), document_canonical_url(soup, rec.get("source_url") or ""))
            common["html_parse_ms"] = (perf_counter() - started) * 1000
        except Exception as exc:
            log.extend(dict(common, variant=v, extraction_status=stage, error_type=type(exc).__name__,
                            error_message=str(exc), error_traceback=traceback.format_exc()) for v in variants)
            print(f"[warn] {rec['html_file']}: {type(exc).__name__}: {exc}")
            continue
        for variant in variants:
            started = perf_counter()
            record = dict(common, variant=variant, extraction_status="ok", error_type="", error_message="", error_traceback="")
            try:
                candidates = EXTRACTORS[variant](html_text, soup, page_url)
                record["candidate_count"] = len(candidates)
                rows = deduplicate_predictions(candidates)
                record["deduplicated_count"] = len(rows)
                record["candidate_limit_hit"] = len(rows) > MAX_CANDIDATES_PER_HTML
                pending = []
                for row in rows[:MAX_CANDIDATES_PER_HTML]:
                    url = row.get("product_url")
                    if not url:
                        continue
                    resolved = urljoin(page_url or rec.get("source_url") or "", str(url))
                    product_url = _norm_url(resolved)
                    if not product_url:
                        continue
                    pending.append({
                        "dataset": rec["dataset"], "page_id": rec["page_id"], "html_file": rec["html_file"],
                        "html_key": rec["html_key"], "run_id": rec["run_id"], "source": rec["source"],
                        "product_url": product_url, "variant": variant,
                        "pred_name": _clean_text(row.get("pred_name")) or None,
                        "pred_price": parse_price(row.get("pred_price")), "pred_in_stock": row.get("pred_in_stock"),
                        "extract_notes": row.get("extract_notes", ""), "html_path": rec["html_path"],
                    })
                out.extend(pending)
                record["returned_count"] = len(pending)
            except Exception as exc:
                record.update(extraction_status="extract_error", returned_count=0, error_type=type(exc).__name__,
                              error_message=str(exc), error_traceback=traceback.format_exc())
                print(f"[warn] {rec['html_file']} {variant}: {type(exc).__name__}: {exc}")
            record["extract_ms"] = (perf_counter() - started) * 1000
            log.append(record)
    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "product_url", "variant",
               "pred_name", "pred_price", "pred_in_stock", "extract_notes", "html_path"]
    pred = pd.DataFrame(out, columns=columns)
    if not pred.empty:
        pred = pred.sort_values(["dataset", "variant", "html_key", "product_url"]).reset_index(drop=True)
    pred.attrs["extraction_log"] = log
    return pred


ORIGINAL_VARIANTS = ("original_fast", "original_fast_query")


def original_parser_contract(variants):
    if not set(variants) & set(ORIGINAL_VARIANTS):
        return None
    return {"bundle_sha256": ORIGINAL_PARSER_META["bundle_sha256"],
            "worker_sha256": ORIGINAL_PARSER_META["worker_sha256"],
            "limit": 500, "source_selectors": "built-in profiles only", "variants": list(variants)}


def prepare_original_runtime():
    import os
    import shutil
    import subprocess
    import zlib

    node = shutil.which("node")
    if not node:
        raise RuntimeError("Original parser requires Node.js (reference: 24.6.0). Install Node.js and restart the kernel. npm and PriceTracker are not required.")
    env = {k: v for k, v in os.environ.items() if k.upper() not in {"NODE_PATH", "NODE_OPTIONS"}}
    version = subprocess.check_output([node, "--version"], text=True, env=env).strip()
    if int(version.lstrip("v").split(".")[0]) < 24:
        raise RuntimeError(f"Use Node.js 24 or newer; reference runtime is 24.6.0, found {version}")
    bundle = zlib.decompress((ORIGINAL_PARSER_ASSET_DIR / "original_fast.cjs.zlib").read_bytes())
    if hashlib.sha256(bundle).hexdigest() != ORIGINAL_PARSER_META["bundle_sha256"]:
        raise ValueError("Embedded original parser bundle failed SHA256 verification")
    worker = (ORIGINAL_PARSER_ASSET_DIR / "worker.cjs").read_bytes()
    # Git on Windows may check text assets out with CRLF. Hash and execute the canonical LF payload.
    worker = worker.replace(b"\r\n", b"\n")
    if hashlib.sha256(worker).hexdigest() != ORIGINAL_PARSER_META["worker_sha256"]:
        raise ValueError("Original worker asset failed SHA256 verification after newline normalization")
    token = ORIGINAL_PARSER_META["bundle_sha256"][:16] + "-" + ORIGINAL_PARSER_META["worker_sha256"][:12]
    runtime = Path(ORIGINAL_RUNTIME_DIR) / token
    runtime.mkdir(parents=True, exist_ok=True)
    payloads = {"original_fast.cjs": bundle, "worker.cjs": worker}
    for name, source in ORIGINAL_SOURCE_FILES.items():
        data = source.encode("utf-8")
        if hashlib.sha256(data).hexdigest() != ORIGINAL_PARSER_META["source_sha256"][name]:
            raise ValueError(f"Embedded original source failed SHA256 verification: {name}")
        payloads[name] = data
    payloads["THIRD_PARTY_LICENSES.json"] = (ORIGINAL_PARSER_ASSET_DIR / "THIRD_PARTY_LICENSES.json").read_bytes()
    for name, data in payloads.items():
        path = runtime / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file() or path.read_bytes() != data:
            path.write_bytes(data)
    return node, version, runtime, env


def run_original_baselines(manifest: pd.DataFrame, variants=ORIGINAL_VARIANTS) -> pd.DataFrame:
    import shutil
    import subprocess

    columns = ["dataset", "page_id", "html_file", "html_key", "run_id", "source", "product_url", "variant",
               "pred_name", "pred_price", "pred_in_stock", "extract_notes", "html_path"]
    if manifest.empty:
        result = pd.DataFrame(columns=columns)
        result.attrs["extraction_log"] = []
        return result
    if manifest.dataset.nunique() != 1 or manifest.page_id.duplicated().any():
        raise ValueError("Original extraction needs one dataset and unique page IDs")
    dataset = str(manifest.dataset.iloc[0])
    node, version, runtime, env = prepare_original_runtime()
    pages = []
    for page in manifest.to_dict("records"):
        if not _is_blank(page.get("skip_reason")):
            continue
        request = {k: _clean_text(page.get(k)) for k in ["page_id", "html_key", "source", "page_url", "source_url", "query"]}
        request["html_path"] = str(Path(page["html_path"]).resolve())
        request["content_sha256"] = _clean_text(page.get("content_sha256")) or hashlib.sha256(Path(request["html_path"]).read_bytes()).hexdigest()
        pages.append(request)
    contract = original_parser_contract(variants)
    signature = hashlib.sha256(json.dumps({"requests": pages, "contract": contract, "node": version},
                                          sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    checkpoint = Path(ORIGINAL_RUNTIME_DIR) / "checkpoints" / dataset / signature
    checkpoint.mkdir(parents=True, exist_ok=True)
    inputs, responses = checkpoint / "inputs.json", checkpoint / "responses.jsonl"
    inputs.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
    cached_ids = set()
    if RESUME_EXTRACTION and responses.is_file():
        for line in responses.read_text(encoding="utf-8").splitlines():
            try:
                cached_ids.add(json.loads(line)["page_id"])
            except json.JSONDecodeError:
                break  # Worker recovers an incomplete final write; earlier corruption is an error.
    command = [node, str(runtime / "worker.cjs"), str(inputs.resolve()), str(responses.resolve())]
    if RESUME_EXTRACTION:
        command.append("--resume")
    with subprocess.Popen(command, cwd=runtime, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(f"[{dataset}] {line.rstrip()}", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Original parser process failed; checkpoints: {checkpoint}")
    records = [json.loads(line) for line in responses.read_text(encoding="utf-8").splitlines() if line]
    if len(records) != len(pages) or {r["page_id"] for r in records} != {p["page_id"] for p in pages}:
        raise ValueError("Original parser returned incomplete or duplicate page results")
    out = OUTPUT_DIR / dataset
    out.mkdir(parents=True, exist_ok=True)
    response_path = out / "original_parser_responses.jsonl"
    shutil.copyfile(responses, response_path)
    errors = [r for r in records if r["status"] != "ok"]
    if errors:
        raise RuntimeError(f"Original parser failed on {len(errors)} pages; see {response_path}: {errors[0].get('error')}")
    by_page = {r["page_id"]: r for r in records}
    predictions, logs = [], []
    for page in manifest.to_dict("records"):
        common = {k: page.get(k) for k in ["dataset", "page_id", "html_key", "html_file", "source", "skip_reason"]}
        common["benchmark_run_id"] = BENCHMARK_RUN_ID
        if not _is_blank(page.get("skip_reason")):
            logs.extend(dict(common, variant=v, extraction_status="skipped") for v in variants)
            continue
        record = by_page[page["page_id"]]
        cached = page["page_id"] in cached_ids
        for variant in variants:
            rows = record["result"]["rows"] if variant == "original_fast" else record["queried"]["rows"]
            candidates = [dict(product_url=r["product_url"], pred_name=r["name"], pred_price=r["price"],
                               pred_in_stock=r.get("in_stock"), extract_notes=record["result"]["route"]) for r in rows]
            deduplicated = deduplicate_predictions(candidates)
            for candidate in deduplicated:
                predictions.append(dict({k: page[k] for k in ["dataset", "page_id", "html_file", "html_key", "run_id", "source"]},
                                        variant=variant, html_path=page["html_path"], **candidate))
            logs.append(dict(common, variant=variant, extraction_status="ok", original_status=record["result"]["status"],
                             replay_cached=cached, html_read_ms=None if cached else record["html_read_ms"], html_parse_ms=None,
                             extract_ms=None if cached else record["extract_ms"] + (record["query_filter_ms"] if variant.endswith("_query") else 0),
                             shared_extraction_ms=None if cached else record["extract_ms"],
                             query_filter_ms=None if cached else record["query_filter_ms"], timing_scope="HTML parse + extraction; shared across modes",
                             candidate_count=len(rows), deduplicated_count=len(deduplicated), returned_count=len(deduplicated),
                             candidate_limit_hit=len(record["result"]["rows"]) >= 500, error_type="", error_message="", error_traceback=""))
    result = pd.DataFrame(predictions, columns=columns)
    result.attrs["extraction_log"] = logs
    result.attrs["original_responses_path"] = str(response_path)
    result.attrs["original_parser"] = dict(ORIGINAL_PARSER_META, node_version=version,
                                          input_fingerprint=signature, resumed_pages=len(cached_ids),
                                          parser_contract=contract)
    return result


def run_baselines(manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    variants = tuple(variants)
    original = tuple(v for v in variants if v in ORIGINAL_VARIANTS)
    legacy = tuple(v for v in variants if v not in ORIGINAL_VARIANTS)
    if not original:
        return run_legacy_baselines(manifest, legacy)
    result = run_original_baselines(manifest, original)
    if legacy:
        old = run_legacy_baselines(manifest, legacy)
        attributes = dict(result.attrs)
        attributes["extraction_log"] = [*attributes["extraction_log"], *old.attrs.get("extraction_log", [])]
        result = pd.concat([result, old], ignore_index=True)
        result.attrs = attributes
    return result


EXTRACTION_LOG_COLUMNS += ["original_status", "replay_cached", "shared_extraction_ms", "query_filter_ms", "timing_scope"]


## 6. Page-scoped evaluation

The matching key is (relative HTML path, normalized product URL) within a dataset.
Both predictions and GT are restricted to available, fully annotated pages in
the supplied manifest, so evaluating a class/cluster slice does not add misses
from other pages. Empty pages require a negative GT annotation (is_product=0).

Results include matched_gt, missing_prediction and unmatched_prediction rows.
GT identifiers remain present for misses; unmatched predictions have no GT ID.
result_id identifies every result row; pred_id is empty for a missing prediction.
Uniqueness is checked per method. Additional classifications can be joined using
(dataset, entity_id), or (dataset, page_id) for page-level classes.

Known absent prices are not numeric price targets. Predicting a price for such
a matched product counts as price_FP/unexpected_price_FP. Unknown GT prices
remain unscored. price_presence_ok additionally checks presence/absence on
matched products. Per-row entity/name/price TP/FP/FN support grouped metrics.


In [8]:
def name_matches(pred: object, truth: object, threshold: float = NAME_THRESHOLD) -> bool:
    return False if _is_blank(pred) or _is_blank(truth) else _name_match(str(pred), str(truth), threshold)


def price_matches(pred: object, truth: object, rel_tol: float = PRICE_REL_TOL) -> bool:
    p = parse_price(pred)
    return False if p is None or _is_blank(truth) else _price_match(p, truth, rel_tol)


def stable_pred_id(dataset: str, variant: str, html_key: str, product_key: str) -> str:
    raw = "\x1f".join([dataset, variant, html_key, product_key]).encode("utf-8")
    return f"{variant}-{hashlib.sha1(raw).hexdigest()[:14]}"


def collapse_prediction_keys(pred: pd.DataFrame) -> pd.DataFrame:
    if pred.empty:
        return pred.copy()
    pred = pred.copy()
    # Also normalize cached/external predictions before joining to the current GT.
    pred["html_key"] = pred.html_key.map(normalize_html_key)
    pred["product_url"] = pred.product_url.map(_norm_url)
    pred = pred[pred.product_url.notna()]
    rows = []
    for _, group in pred.groupby(["variant", "html_key", "product_url"], sort=False):
        records = sorted(group.to_dict("records"), key=candidate_completeness, reverse=True)
        rows.append(records[0])
    return pd.DataFrame(rows, columns=pred.columns)


def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, 2 * p * r / (p + r) if p + r else 0.0


COUNT_COLUMNS = [f"{field}_{outcome}" for field in ("entity", "name", "price") for outcome in ("TP", "FP", "FN")]
EVALUATION_COLUMNS = [
    "benchmark_run_id", "result_id", "pred_id", "dataset", "variant", "page_id", "gt_id", "entity_id",
    "html_file", "html_key", "run_id", "source", "query", "product_url", "match_status",
    "gt_name", "gt_price", "gt_price_state", "pred_name", "pred_price", "name_ok", "price_ok",
    "price_presence_ok", "unexpected_price_FP", "name_similarity", "price_error_signed", "price_error_absolute",
    "price_error_relative", "price_error_relative_absolute", "error_labels", "notes", *COUNT_COLUMNS,
    "_gt_exists", "_gt_name_present", "_gt_price_present", "_pred_name_present", "_pred_price_present",
]
EXPORT_EVALUATION_COLUMNS = [c for c in EVALUATION_COLUMNS if not c.startswith("_")]
PAGE_RESULT_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "page_id", "html_file", "query", "page_status", "is_scored",
    "gt_positive_count", "prediction_count", "matched_gt", "missing_prediction", "unmatched_prediction",
    "page_has_products_ok", *COUNT_COLUMNS,
]


def evaluate_predictions(raw_pred: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                         variants=VARIANTS) -> tuple[pd.DataFrame, pd.DataFrame]:
    variants = tuple(variants)
    if not variants or len(set(variants)) != len(variants):
        raise ValueError("Evaluation variants must be nonempty and unique")
    if not manifest.empty and manifest.dataset.nunique() != 1:
        raise ValueError("Evaluate one dataset at a time")
    dataset_name = str(manifest.dataset.iloc[0]) if not manifest.empty else (
        str(raw_pred.dataset.iloc[0]) if not raw_pred.empty else "dataset")
    gt = align_groundtruth_to_manifest(gt, manifest)
    scoped_gt, scored_manifest = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive].copy()
    for column in ["gt_id", "entity_id"]:
        if positive[column].map(_is_blank).any() or positive[column].astype(str).duplicated().any():
            raise ValueError(f"Positive GT requires unique nonempty {column}")
    if positive.duplicated(["html_key", "_eval_product_key"]).any():
        raise ValueError("Duplicate positive GroundTruth page/URL key after URL normalization")
    negative_keys = set(scoped_gt.loc[~scoped_gt._positive, "html_key"])
    if negative_keys & set(positive.html_key):
        raise ValueError("A page has both positive and negative GT rows")
    pred = collapse_prediction_keys(raw_pred)
    if not pred.empty:
        pred = pred[pred.html_key.isin(scored_manifest.html_key) & pred.variant.isin(variants)
                    & pred.dataset.eq(dataset_name)]
    gt_map = {(r.html_key, str(r._eval_product_key)): r for _, r in positive.iterrows()}
    page_map = scored_manifest.set_index("html_key").to_dict("index")
    rows = []
    for variant in variants:
        pv = pred[pred.variant == variant] if not pred.empty else pred
        pred_map = {(r["html_key"], str(r["product_url"])): r for r in pv.to_dict("records")}
        for html_key, product_key in sorted(set(gt_map) | set(pred_map)):
            p = pred_map.get((html_key, product_key))
            g = gt_map.get((html_key, product_key))
            meta = page_map[html_key]
            gt_exists, pred_exists = g is not None, p is not None
            pred_name = (_clean_text(p.get("pred_name")) or None) if pred_exists else None
            pred_price = parse_price(p.get("pred_price")) if pred_exists else None
            gt_name = g.gt_name if gt_exists else None
            gt_price = g.gt_price if gt_exists else None
            state = g.gt_price_state if gt_exists else None
            has_gt_name = gt_exists and not _is_blank(gt_name)
            has_pred_name = not _is_blank(pred_name)
            has_pred_price = pred_price is not None
            name_ok = int(name_matches(pred_name, gt_name)) if has_gt_name else (0 if has_pred_name and not gt_exists else None)
            price_ok = int(price_matches(pred_price, gt_price)) if state == "present" else (
                0 if has_pred_price and (not gt_exists or state == "absent") else None)
            presence_ok = (int(has_pred_price == (state == "present"))
                           if gt_exists and pred_exists and state in {"present", "absent"} else None)
            status = "matched_gt" if gt_exists and pred_exists else (
                "missing_prediction" if gt_exists else "unmatched_prediction")
            result_id = stable_pred_id(dataset_name, variant, html_key, product_key)
            notes = status
            if pred_exists and _clean_text(p.get("extract_notes")):
                notes += "; " + _clean_text(p["extract_notes"])
            rows.append({
                "benchmark_run_id": BENCHMARK_RUN_ID, "result_id": result_id,
                "pred_id": result_id if pred_exists else None, "dataset": dataset_name, "variant": variant,
                "page_id": meta["page_id"], "gt_id": g.gt_id if gt_exists else None,
                "entity_id": g.entity_id if gt_exists else None,
                "html_file": meta["html_file"], "html_key": html_key,
                "run_id": meta["run_id"], "source": meta["source"],
                "query": _first_nonempty(g.query if gt_exists else None, meta.get("query")),
                "product_url": g.product_url if gt_exists else p["product_url"], "match_status": status,
                "gt_name": gt_name, "gt_price": gt_price, "gt_price_state": state,
                "pred_name": pred_name, "pred_price": pred_price, "name_ok": name_ok, "price_ok": price_ok,
                "price_presence_ok": presence_ok,
                "unexpected_price_FP": int(gt_exists and state == "absent" and has_pred_price), "notes": notes,
                "entity_TP": int(gt_exists and pred_exists), "entity_FP": int(not gt_exists),
                "entity_FN": int(gt_exists and not pred_exists),
                "name_TP": int(has_gt_name and name_ok == 1),
                "name_FP": int(has_pred_name and (not gt_exists or name_ok == 0)),
                "name_FN": int(has_gt_name and name_ok != 1),
                "price_TP": int(state == "present" and price_ok == 1),
                "price_FP": int(has_pred_price and (not gt_exists or state == "absent" or price_ok == 0)),
                "price_FN": int(state == "present" and price_ok != 1),
                "_gt_exists": gt_exists, "_gt_name_present": has_gt_name, "_gt_price_present": state == "present",
                "_pred_name_present": has_pred_name, "_pred_price_present": has_pred_price,
            })
    for row in rows:
        name_pair = row["_gt_name_present"] and row["_pred_name_present"]
        row["name_similarity"] = (SequenceMatcher(None, row["pred_name"].lower().strip(),
                                                  row["gt_name"].lower().strip()).ratio() if name_pair else None)
        price_pair = row["_gt_price_present"] and row["_pred_price_present"]
        error = row["pred_price"] - row["gt_price"] if price_pair else None
        relative = error / abs(row["gt_price"]) if price_pair and row["gt_price"] != 0 else None
        row.update(price_error_signed=error, price_error_absolute=abs(error) if error is not None else None,
                   price_error_relative=relative,
                   price_error_relative_absolute=abs(relative) if relative is not None else None)
        labels = []
        if row["entity_FN"]: labels.append("missing_entity")
        if row["entity_FP"]: labels.append("unexpected_entity")
        if row["match_status"] == "matched_gt":
            if row["name_ok"] == 0: labels.append("wrong_name" if row["_pred_name_present"] else "missing_name")
            if row["gt_price_state"] == "present" and row["price_ok"] == 0:
                labels.append("wrong_price" if row["_pred_price_present"] else "missing_price")
            if row["unexpected_price_FP"]: labels.append("unexpected_price")
        row["error_labels"] = ";".join(labels)
    evaluation = pd.DataFrame(rows, columns=EVALUATION_COLUMNS)
    for col in [c for c in EVALUATION_COLUMNS if c.startswith("_")]:
        evaluation[col] = evaluation[col].astype(bool)
    for col in [*COUNT_COLUMNS, "unexpected_price_FP"]:
        evaluation[col] = evaluation[col].astype("int64")
    # Nullable integer GT IDs must not become "123.0" when mixed with unmatched predictions.
    evaluation["gt_id"] = pd.Series([r["gt_id"] for r in rows], dtype=object)
    metric_rows = []
    for variant in variants:
        ev = evaluation[evaluation.variant == variant]
        counts = {c: int(ev[c].sum()) if len(ev) else 0 for c in COUNT_COLUMNS}
        row = {"benchmark_run_id": BENCHMARK_RUN_ID, "dataset": dataset_name, "variant": variant,
               "n_pages": len(scored_manifest), "n": len(positive),
               "n_priced": int(positive.gt_price_state.eq("present").sum()),
               "n_price_absent": int(positive.gt_price_state.eq("absent").sum()),
               "n_price_unannotated": int(positive.gt_price_state.eq("unannotated").sum()),
               "unexpected_price_FP": int(ev.unexpected_price_FP.sum()) if len(ev) else 0, **counts}
        for field in ["entity", "name", "price"]:
            p, r, f1 = _prf(*(counts[f"{field}_{outcome}"] for outcome in ["TP", "FP", "FN"]))
            row.update({f"{field}_P": p, f"{field}_R": r, f"{field}_F1": f1})
        metric_rows.append(row)
    return evaluation, pd.DataFrame(metric_rows)


def build_page_results(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                       variants=VARIANTS) -> pd.DataFrame:
    scoped_gt, scored = evaluation_scope(gt, manifest)
    counts = scoped_gt[scoped_gt._positive].groupby("html_key").size().to_dict()
    scored_keys = set(scored.html_key)
    excluded = set(gt.loc[gt._excluded, "html_key"])
    grouped = {(v, key): group for (v, key), group in evaluation.groupby(["variant", "html_key"])}
    rows = []
    for variant in variants:
        for meta in manifest.to_dict("records"):
            key = meta["html_key"]
            ev = grouped.get((variant, key), evaluation.iloc[:0])
            is_scored = key in scored_keys
            n_gt = int(counts.get(key, 0))
            n_pred = int(ev.match_status.ne("missing_prediction").sum())
            status = _first_nonempty(meta.get("skip_reason")) or (
                "excluded_gt" if key in excluded else "unannotated" if not is_scored else
                "positive" if n_gt else "negative")
            rows.append({
                "benchmark_run_id": BENCHMARK_RUN_ID, "dataset": meta["dataset"], "variant": variant,
                "page_id": meta["page_id"], "html_file": meta["html_file"], "query": meta.get("query", ""),
                "page_status": status, "is_scored": is_scored,
                "gt_positive_count": n_gt if is_scored else None, "prediction_count": n_pred if is_scored else None,
                "matched_gt": int(ev.match_status.eq("matched_gt").sum()),
                "missing_prediction": int(ev.match_status.eq("missing_prediction").sum()),
                "unmatched_prediction": int(ev.match_status.eq("unmatched_prediction").sum()),
                "page_has_products_ok": int((n_gt > 0) == (n_pred > 0)) if is_scored else None,
                **{c: int(ev[c].sum()) if len(ev) else 0 for c in COUNT_COLUMNS},
            })
    pages = pd.DataFrame(rows, columns=PAGE_RESULT_COLUMNS)
    pages["is_scored"] = pages.is_scored.astype(bool)
    return pages


## 7. Exports

Outputs are written to baseline_results/determined/<dataset>/:

- baseline_predictions.csv: explicit GT IDs, match_status and per-row counters;
- baseline_page_results.csv: all manifest pages, including negative/unscored/missing pages;
- baseline_metrics.csv: entity, name and price metrics;
- baseline_entity_analytics.csv: GT/Pages metadata, class/cluster labels and numeric errors;
- baseline_group_metrics.csv: per-site/query/page-type/class metrics and page confusion;
- baseline_error_breakdown.csv, baseline_error_distributions.csv: chart-ready error counts;
- baseline_extraction_log.csv: measured timings, candidate counts, exceptions and skipped files;
- baseline_analysis_profile.json: distributions, table schemas and metric definitions;
- baseline_raw_predictions.csv, groundtruth_loaded.csv, html_manifest.csv;
- groundtruth_audit.json and benchmark_run.json: scope, GT fingerprint and settings.

Additional GT classification columns are preserved in groundtruth_loaded.csv.
Join by dataset/entity_id for products and dataset/page_id for pages. Include
benchmark_run_id when combining multiple executions. Empty exports keep their
column schema. See BENCHMARK.md for the join contract and price-state semantics.

- baseline_review.xlsx: GT and method names/prices side by side, extra predictions, metrics and page diagnostics.
- baseline_review_all.xlsx in the output root combines all included datasets.


In [9]:

DISPLAY_NAMES = {
    "original_fast": "Original fast-renderer",
    "original_fast_query": "Original + query filter",
    "b1_jsonld": "B1 JSON-LD",
    "b2_css": "B2 CSS heuristics",
    "b3_regex": "B3 Regex/text",
}

METRIC_COLUMNS = [
    "benchmark_run_id", "dataset", "variant", "n_pages", "n", "n_priced", "n_price_absent", "n_price_unannotated",
    "entity_TP", "entity_FP", "entity_FN", "entity_P", "entity_R", "entity_F1", "unexpected_price_FP",
    "name_TP", "name_FP", "name_FN", "name_P", "name_R", "name_F1",
    "price_TP", "price_FP", "price_FN", "price_P", "price_R", "price_F1",
]


def export_dataset_outputs(
    dataset_name: str,
    manifest: pd.DataFrame,
    raw_pred: pd.DataFrame,
    gt: pd.DataFrame | None,
    audit: dict | None,
    evaluation: pd.DataFrame | None,
    metrics: pd.DataFrame | None,
) -> dict[str, Path]:
    out = OUTPUT_DIR / dataset_name
    out.mkdir(parents=True, exist_ok=True)

    paths = {}
    manifest_path = out / "html_manifest.csv"
    raw_path = out / "baseline_raw_predictions.csv"
    manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
    raw_pred.to_csv(raw_path, index=False, encoding="utf-8-sig")
    paths["manifest"] = manifest_path
    paths["raw_predictions"] = raw_path

    if gt is not None:
        gt_export = gt.drop(columns=[c for c in gt.columns if c.startswith("_")], errors="ignore")
        gt_path = out / "groundtruth_loaded.csv"
        gt_export.to_csv(gt_path, index=False, encoding="utf-8-sig")
        paths["groundtruth_loaded"] = gt_path
    else:
        gt_path = out / "groundtruth_loaded.csv"
        pd.DataFrame(columns=["gt_id", "entity_id", "page_id", "html_file"]).to_csv(gt_path, index=False, encoding="utf-8-sig")
        paths["groundtruth_loaded"] = gt_path

    if audit is not None:
        audit_path = out / "groundtruth_audit.json"
        audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
        paths["audit"] = audit_path
    else:
        audit_path = out / "groundtruth_audit.json"
        audit_path.write_text(json.dumps({"evaluation_status": "not_scored", "reason": "audit not supplied"}), encoding="utf-8")
        paths["audit"] = audit_path

    if evaluation is not None:
        pred_path = out / "baseline_predictions.csv"
        evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).to_csv(pred_path, index=False, encoding="utf-8-sig", na_rep="")
        paths["predictions"] = pred_path
    else:
        pred_path = out / "baseline_predictions.csv"
        pd.DataFrame(columns=EXPORT_EVALUATION_COLUMNS).to_csv(pred_path, index=False, encoding="utf-8-sig")
        paths["predictions"] = pred_path

    if metrics is not None:
        metrics_path = out / "baseline_metrics.csv"
        metrics[METRIC_COLUMNS].to_csv(metrics_path, index=False, encoding="utf-8-sig")
        paths["metrics"] = metrics_path
    else:
        metrics_path = out / "baseline_metrics.csv"
        pd.DataFrame(columns=METRIC_COLUMNS).to_csv(metrics_path, index=False, encoding="utf-8-sig")
        paths["metrics"] = metrics_path

    if raw_pred.attrs.get("original_responses_path"):
        paths["original_responses"] = Path(raw_pred.attrs["original_responses_path"])
    paths.update(export_analysis_outputs(out, manifest, raw_pred, gt, evaluation, metrics))
    paths["review_excel"] = export_review_workbook(
        out / "baseline_review.xlsx",
        evaluation if evaluation is not None else pd.DataFrame(columns=EXPORT_EVALUATION_COLUMNS),
        metrics if metrics is not None else pd.DataFrame(columns=METRIC_COLUMNS),
        pd.read_csv(paths["page_results"]), manifest,
        provenance={"dataset": dataset_name, "benchmark_run_id": BENCHMARK_RUN_ID, "original_parser": raw_pred.attrs.get("original_parser")})
    metadata = {
        "schema_version": 2, "benchmark_run_id": BENCHMARK_RUN_ID,
        "dataset": dataset_name, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "reference_file": gt.attrs.get("source_path") if gt is not None else None,
        "reference_sha256": gt.attrs.get("reference_sha256") if gt is not None else None,
        "annotation_policy": gt.attrs.get("annotation_policy") if gt is not None else ANNOTATION_POLICY,
        "industrial_zero_means_absent_price": gt.attrs.get("zero_price_is_missing") if gt is not None else None,
        "variants": list(metrics.variant) if metrics is not None else list(VARIANTS),
        "name_threshold": NAME_THRESHOLD, "price_relative_tolerance": PRICE_REL_TOL,
        "url_policy": "lowercase; strip query, fragment, trailing slash and www host prefix",
        "manifest_selection": manifest.attrs.get("selection"),
        "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
        "raw_predictions_sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
        "pandas_version": pd.__version__,
        "python_version": __import__("platform").python_version(),
        "parser_contract": original_parser_contract(tuple(metrics.variant) if metrics is not None else VARIANTS),
        "original_parser": raw_pred.attrs.get("original_parser"),
        "evaluation_status": "scored" if evaluation is not None else "not_scored",
        "files": {key: {"file": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
                  for key, path in paths.items()},
    }
    notebook_path = BENCHMARK_ROOT / "baseline_determined.ipynb"
    if notebook_path.exists():
        metadata["notebook_sha256"] = hashlib.sha256(notebook_path.read_bytes()).hexdigest()
    metadata_path = out / "benchmark_run.json"
    metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["run_metadata"] = metadata_path

    return paths


def paper_table(all_metrics: pd.DataFrame) -> pd.DataFrame:
    if all_metrics.empty:
        return all_metrics
    out = all_metrics.copy()
    out["Method"] = out["variant"].map(DISPLAY_NAMES).fillna(out["variant"])
    return out[
        ["dataset", "Method", "n_pages", "n",
         "name_P", "name_R", "name_F1", "price_P", "price_R", "price_F1"]
    ].rename(columns={"dataset": "Dataset"})


def extraction_log_frame(raw_pred: pd.DataFrame, manifest: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    records = raw_pred.attrs.get("extraction_log")
    if records is None:
        # Cached predictions do not contain measured timings; never invent them.
        records = [dict(benchmark_run_id=BENCHMARK_RUN_ID, dataset=m["dataset"], variant=v,
                        page_id=m["page_id"], html_key=m["html_key"], html_file=m["html_file"], source=m["source"],
                        skip_reason=m.get("skip_reason", ""),
                        extraction_status="skipped" if not _is_blank(m.get("skip_reason")) else "not_recorded")
                   for m in manifest.to_dict("records") for v in variants]
    return pd.DataFrame(records, columns=EXTRACTION_LOG_COLUMNS)


def _page_metadata(manifest: pd.DataFrame) -> pd.DataFrame:
    return manifest.rename(columns={c: c if c == "page_id" or c.startswith("page_meta__") else "manifest__" + c
                                    for c in manifest.columns})


def build_entity_analytics(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame) -> pd.DataFrame:
    result = evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).copy()
    labels = gt.loc[gt._positive.astype(bool), [c for c in gt.columns if not c.startswith("_")]].copy()
    labels = labels.rename(columns={c: "gt_meta__" + c for c in labels.columns if c != "entity_id"})
    # Remove unscored duplicate IDs from unrelated pages by restricting to the actual evaluation scope.
    labels = labels[labels.entity_id.isin(result.entity_id.dropna())]
    result = result.merge(labels, on="entity_id", how="left", validate="many_to_one")
    return result.merge(_page_metadata(manifest), on="page_id", how="left", validate="many_to_one")


def build_page_analytics(evaluation: pd.DataFrame, gt: pd.DataFrame, manifest: pd.DataFrame,
                         raw_pred: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    pages = build_page_results(evaluation, gt, manifest, variants)
    pages = pages.merge(_page_metadata(manifest), on="page_id", how="left", validate="many_to_one")
    for col in ["source", "run_id", "location", "route"]:
        pages[col] = pages.get("manifest__" + col, "")
    log = extraction_log_frame(raw_pred, manifest, variants)
    log_columns = ["variant", "page_id", *[c for c in EXTRACTION_LOG_COLUMNS if c not in
                   {"variant", "page_id", "benchmark_run_id", "dataset", "html_key", "html_file", "source", "skip_reason"}]]
    pages = pages.merge(log[log_columns], on=["variant", "page_id"], how="left", validate="one_to_one")
    scoped_gt, _ = evaluation_scope(gt, manifest)
    positive = scoped_gt[scoped_gt._positive]
    for state in ["present", "absent", "unannotated"]:
        count = positive[positive.gt_price_state.eq(state)].groupby("html_key").size()
        pages["gt_price_" + state + "_count"] = pages.manifest__html_key.map(count).fillna(0).astype(int)
    matched = evaluation[evaluation.match_status.eq("matched_gt")]
    diagnostics = matched.groupby(["variant", "page_id"]).agg(
        matched_name_similarity_mean=("name_similarity", "mean"),
        matched_price_MAE=("price_error_absolute", "mean"),
        matched_price_MAPE=("price_error_relative_absolute", "mean"),
        unexpected_price_FP=("unexpected_price_FP", "sum"),
    ).reset_index()
    pages = pages.merge(diagnostics, on=["variant", "page_id"], how="left", validate="one_to_one")
    pages["unexpected_price_FP"] = pages.unexpected_price_FP.fillna(0).astype(int)
    actual = pages.gt_positive_count.fillna(0).gt(0)
    predicted = pages.prediction_count.fillna(0).gt(0)
    for label, mask in {"TP": actual & predicted, "FP": ~actual & predicted,
                        "FN": actual & ~predicted, "TN": ~actual & ~predicted}.items():
        pages["page_" + label] = (pages.is_scored & mask).astype(int)
    for field in ["entity", "name", "price"]:
        values = [_prf(int(tp), int(fp), int(fn)) for tp, fp, fn in
                  zip(pages[field + "_TP"], pages[field + "_FP"], pages[field + "_FN"])]
        for i, metric in enumerate(["P", "R", "F1"]):
            pages[field + "_" + metric] = [v[i] if scored else None for v, scored in zip(values, pages.is_scored)]
    return pages


def build_group_metrics(entities: pd.DataFrame, pages: pd.DataFrame) -> pd.DataFrame:
    columns = ["benchmark_run_id", "dataset", "variant", "group_level", "group_dimension", "group_value",
               "n_pages", "n_entities", "n_predictions", "n_priced", "n_price_absent", "unexpected_price_FP",
               *COUNT_COLUMNS,
               *[f"{f}_{m}" for f in ["entity", "name", "price"] for m in ["P", "R", "F1"]],
               "page_TP", "page_FP", "page_FN", "page_TN", "page_accuracy",
               "macro_page_entity_F1", "macro_page_name_F1", "macro_page_price_F1"]
    rows = []

    def class_dimension(name):
        return any(part in name.casefold() for part in ["category", "class", "cluster", "label"])

    page_dims = [c for c in ["source", "page_status", "query", "location", "route"] if c in pages]
    page_dims += [c for c in pages if c.startswith("page_meta__") and (
        class_dimension(c) or c in {"page_meta__page_kind", "page_meta__metadata_is_blocked", "page_meta__empty_result_reason", "page_meta__mutation_axis", "page_meta__base_template", "page_meta__variant_idx"})]
    entity_dims = ["gt_price_state"] + [c for c in entities if c.startswith("gt_meta__") and class_dimension(c)]
    for level, frame, dimensions in [("page", pages[pages.is_scored], page_dims), ("entity", entities, entity_dims)]:
        for dim in dimensions:
            grouped_frame = frame.assign(_group_value=frame[dim].map(
                lambda value: "<unmatched_or_unlabeled>" if _is_blank(value) else str(value)))
            for (dataset, variant, value), group in grouped_frame.groupby(["dataset", "variant", "_group_value"], sort=False):
                counts = {c: int(group[c].sum()) for c in COUNT_COLUMNS}
                row = dict(benchmark_run_id=BENCHMARK_RUN_ID, dataset=dataset, variant=variant, group_level=level,
                           group_dimension=dim, group_value="<unmatched_or_unlabeled>" if _is_blank(value) else str(value),
                           n_pages=group.page_id.nunique(), n_entities=counts["entity_TP"] + counts["entity_FN"],
                           n_predictions=counts["entity_TP"] + counts["entity_FP"],
                           n_priced=counts["price_TP"] + counts["price_FN"],
                           n_price_absent=int(group.gt_price_absent_count.sum()) if level == "page" else int(group.gt_price_state.eq("absent").sum()),
                           unexpected_price_FP=int(group.unexpected_price_FP.sum()), **counts)
                for field in ["entity", "name", "price"]:
                    values = _prf(*(counts[f"{field}_{m}"] for m in ["TP", "FP", "FN"]))
                    row.update({f"{field}_{m}": v for m, v in zip(["P", "R", "F1"], values)})
                if level == "page":
                    row.update({"page_" + c: int(group["page_" + c].sum()) for c in ["TP", "FP", "FN", "TN"]})
                    row["page_accuracy"] = (row["page_TP"] + row["page_TN"]) / len(group)
                    row.update({"macro_page_" + f + "_F1": float(group[f + "_F1"].mean()) for f in ["entity", "name", "price"]})
                rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def build_error_distributions(evaluation: pd.DataFrame, variants=VARIANTS) -> pd.DataFrame:
    rows = []
    for variant in variants:
        ev = evaluation[evaluation.variant.eq(variant)]
        for measure, column, edges in [
            ("name_similarity", "name_similarity", [i / 10 for i in range(11)]),
            ("absolute_relative_price_error", "price_error_relative_absolute", [0, .01, .05, .1, .25, .5, 1, float("inf")]),
        ]:
            values = pd.to_numeric(ev[column], errors="coerce").dropna()
            for i, (lower, upper) in enumerate(zip(edges, edges[1:])):
                mask = (values.ge(lower) if i == 0 else values.gt(lower)) & values.le(upper)
                rows.append(dict(benchmark_run_id=BENCHMARK_RUN_ID, variant=variant, measure=measure,
                                 bin_lower=lower, bin_upper=upper if math.isfinite(upper) else None,
                                 lower_inclusive=i == 0, upper_inclusive=True, count=int(mask.sum()),
                                 n_comparable_pairs=len(values)))
    return pd.DataFrame(rows, columns=["benchmark_run_id", "variant", "measure", "bin_lower", "bin_upper",
                                      "lower_inclusive", "upper_inclusive", "count", "n_comparable_pairs"])


def export_analysis_outputs(out: Path, manifest: pd.DataFrame, raw_pred: pd.DataFrame,
                            gt: pd.DataFrame | None, evaluation: pd.DataFrame | None,
                            metrics: pd.DataFrame | None) -> dict[str, Path]:
    variants = tuple(metrics.variant) if metrics is not None else VARIANTS
    ev = evaluation if evaluation is not None else pd.DataFrame(columns=EVALUATION_COLUMNS)
    if gt is None:
        gt = pd.DataFrame(columns=["html_key", "gt_price_state", "gt_price", "entity_id", "gt_id"])
        for col in ["_positive", "_excluded", "_annotated"]:
            gt[col] = pd.Series(dtype=bool)
    log = extraction_log_frame(raw_pred, manifest, variants)
    entities = build_entity_analytics(ev, gt, manifest)
    pages = build_page_analytics(ev, gt, manifest, raw_pred, variants)
    groups = build_group_metrics(entities, pages)
    errors = ev.loc[ev.error_labels.fillna("").ne(""), ["dataset", "variant", "source", "error_labels"]].copy()
    errors["error_type"] = errors.error_labels.str.split(";")
    errors = errors.explode("error_type").groupby(["dataset", "variant", "source", "error_type"]).size().rename("count").reset_index()
    distributions = build_error_distributions(ev, variants)
    distributions.insert(1, "dataset", str(manifest.dataset.iloc[0]) if not manifest.empty else str(out.name))
    tables = {
        "entity_analytics": ("baseline_entity_analytics.csv", entities),
        "page_results": ("baseline_page_results.csv", pages),
        "group_metrics": ("baseline_group_metrics.csv", groups),
        "error_breakdown": ("baseline_error_breakdown.csv", errors),
        "error_distributions": ("baseline_error_distributions.csv", distributions),
        "extraction_log": ("baseline_extraction_log.csv", log),
    }
    paths = {}
    for key, (name, frame) in tables.items():
        path = out / name
        frame.to_csv(path, index=False, encoding="utf-8-sig", na_rep="")
        paths[key] = path
    scoped, _ = evaluation_scope(gt, manifest)
    positive = scoped[scoped._positive]
    price_description = json.loads(pd.to_numeric(positive.gt_price, errors="coerce").describe(
        percentiles=[.1, .25, .5, .75, .9, .95, .99]).to_json())
    timings = log.copy()
    timings["extract_ms"] = pd.to_numeric(timings.extract_ms, errors="coerce")
    timing_summary = json.loads(timings.groupby("variant").extract_ms.describe().reset_index().to_json(orient="records"))
    profile = {
        "benchmark_run_id": BENCHMARK_RUN_ID, "gt_loaded_rows": len(gt), "scored_gt_entities": len(positive),
        "price_description": price_description,
        "scored_page_statuses": pages.loc[pages.is_scored].groupby("variant").page_status.value_counts().rename("count").reset_index().to_dict("records"),
        "extraction_statuses": log.groupby(["variant", "extraction_status"]).size().rename("count").reset_index().to_dict("records"),
        "extraction_time_ms_by_variant": timing_summary,
        "tables": {name: {"rows": len(frame), "columns": list(frame.columns)} for _, (name, frame) in tables.items()},
        "semantics": {
            "name_similarity": "SequenceMatcher on lowercase stripped names; comparable GT/prediction pairs only",
            "price_errors": "Predicted minus GT; relative error is a fraction of abs(GT), undefined for GT=0",
            "conditional_MAE_MAPE": "Computed only where both matched prices exist; use TP/FP/FN for overall quality",
            "error_breakdown": "Error categories can overlap for the same entity; counts are not additive across categories",
            "page_confusion": "Presence of at least one product on a scored page; independent of field correctness",
            "macro_page_F1": "Arithmetic mean over all scored pages in the group; undefined per-page P/R/F1 use zero",
            "gt_meta__": "Original GT columns joined by entity_id; unavailable for unmatched predictions",
            "page_meta__": "Original Pages metadata; manifest__ holds normalized manifest fields",
            "unmatched_or_unlabeled": "Retained group; never silently discard false positives without a GT class",
            "timings": "Milliseconds. HTML read/parse is shared and repeated for each variant; do not sum it across variants. Cached timings remain missing.",
        },
    }
    profile_path = out / "baseline_analysis_profile.json"
    profile_path.write_text(json.dumps(profile, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    paths["analysis_profile"] = profile_path
    return paths


def load_verified_saved_results(dataset_name: str, cfg: dict):
    """Reuse completed unselected datasets, never stale GT or edited result CSVs."""
    folder = OUTPUT_DIR / dataset_name
    metadata_path = folder / "benchmark_run.json"
    if not metadata_path.is_file():
        return None, {"dataset": dataset_name, "status": "missing_saved_results"}
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    reference = cfg.get("reference_file") or discover_dataset_reference(Path(cfg["root"]))
    expected = {
        "schema_version": 2, "variants": list(VARIANTS), "annotation_policy": ANNOTATION_POLICY,
        "parser_contract": original_parser_contract(VARIANTS),
        "name_threshold": NAME_THRESHOLD, "price_relative_tolerance": PRICE_REL_TOL,
        "url_policy": "lowercase; strip query, fragment, trailing slash and www host prefix",
    }
    for key, value in expected.items():
        if metadata.get(key) != value:
            return None, {"dataset": dataset_name, "status": "incompatible_settings", "field": key}
    if reference is None or not Path(reference).is_file() or metadata.get("reference_sha256") != hashlib.sha256(Path(reference).read_bytes()).hexdigest():
        return None, {"dataset": dataset_name, "status": "stale_groundtruth"}
    tables = {}
    for key in ["metrics", "predictions", "raw_predictions", "page_results", "manifest"]:
        item = metadata.get("files", {}).get(key)
        if not item:
            return None, {"dataset": dataset_name, "status": "missing_file_fingerprint", "field": key}
        saved = folder / item["file"]
        if not saved.is_file() or hashlib.sha256(saved.read_bytes()).hexdigest() != item["sha256"]:
            return None, {"dataset": dataset_name, "status": "modified_saved_results", "field": key}
        tables[key] = pd.read_csv(saved, low_memory=False)
    # Filename restoration or edits to any scored snapshot require a fresh run.
    for rec in tables["manifest"].to_dict("records"):
        if not _is_blank(rec.get("skip_reason")):
            continue
        html_path = Path(cfg["html_dir"]) / rec["html_key"]
        if not html_path.is_file() or hashlib.sha256(html_path.read_bytes()).hexdigest() != rec.get("content_sha256"):
            return None, {"dataset": dataset_name, "status": "stale_html", "html_file": rec["html_file"]}
    # The companion manifest defines the split even when the physical directory is shared.
    current_pages = read_pages_metadata(Path(reference))
    if current_pages is not None and set(current_pages) != set(tables["manifest"].html_key):
        return None, {"dataset": dataset_name, "status": "changed_page_selection"}
    return tables, {"dataset": dataset_name, "status": "reused", "benchmark_run_id": metadata["benchmark_run_id"],
                    "reference_sha256": metadata["reference_sha256"], "run_metadata": str(metadata_path)}


# Embedded in notebook cell 16; the notebook has no dependency on this staging file.
def build_review_comparison(evaluation: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One GT entity per row, methods side by side; unmatched predictions stay separate."""
    identity = ["dataset", "entity_id"]
    base_columns = ["dataset", "entity_id", "gt_id", "page_id", "source", "query", "html_file",
                    "product_url", "gt_name", "gt_price", "gt_price_state"]
    method_columns = ["benchmark_run_id", "pred_name", "pred_price", "match_status", "name_similarity",
                      "name_ok", "price_ok", "price_presence_ok", "error_labels", "notes"]
    if evaluation.empty:
        return pd.DataFrame(columns=base_columns), evaluation.copy()
    known = evaluation.loc[evaluation.entity_id.map(lambda v: not _is_blank(v))].copy()
    # CSV reloads turn blank text into NaN and numeric GT IDs into floats.
    # Preserve textual IDs (including leading zeroes), normalizing numeric IDs only.
    for column in base_columns:
        if column in known:
            known[column] = known[column].map(lambda v: None if _is_blank(v) else v)
    for column in ["gt_id", "entity_id", "page_id"]:
        if column in known:
            known[column] = known[column].map(
                lambda v: None if v is None else str(int(v)) if isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer() else str(v))
    if known.duplicated(["dataset", "variant", "entity_id"]).any():
        raise ValueError("Excel comparison requires one run per dataset/method and unique GT entity IDs")
    # GT fields must agree across methods; otherwise side-by-side comparison is misleading.
    base = known.reindex(columns=base_columns).drop_duplicates()
    if base.duplicated(identity).any():
        raise ValueError("Conflicting GT fields across methods in Excel comparison")
    for variant in evaluation.variant.drop_duplicates():
        part = known.loc[known.variant.eq(variant)].reindex(columns=identity + method_columns)
        part = part.rename(columns={c: f"{variant}__{c}" for c in method_columns})
        base = base.merge(part, on=identity, how="left", validate="one_to_one")
    base = base.sort_values(["dataset", "source", "page_id", "entity_id"], kind="stable").reset_index(drop=True)
    # Put business values first; keep technical join keys and diagnostics to the right.
    leading = ["source", "gt_name", "gt_price", "gt_price_state"]
    for variant in evaluation.variant.drop_duplicates():
        leading += [f"{variant}__pred_name", f"{variant}__pred_price"]
    base = base.reindex(columns=leading + [c for c in base if c not in leading])
    extras = evaluation.loc[evaluation.match_status.eq("unmatched_prediction")].copy()
    return base, extras


def export_review_workbook(path: Path, evaluation: pd.DataFrame, metrics: pd.DataFrame,
                           pages: pd.DataFrame | None = None, manifest: pd.DataFrame | None = None,
                           *, provenance: dict | None = None) -> Path:
    """Readable review artifact; numbers and evaluation semantics are unchanged."""
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    from openpyxl.utils import get_column_letter

    if evaluation.empty:
        evaluation = evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS)
    if metrics.empty:
        metrics = metrics.reindex(columns=METRIC_COLUMNS)

    comparison, extras = build_review_comparison(evaluation)

    matched = (
        evaluation.loc[evaluation.match_status.eq("matched_gt")]
        if not evaluation.empty else evaluation
    )
    conditional = []
    for (dataset, variant), group in matched.groupby(["dataset", "variant"], sort=False):
        conditional.append({
            "dataset": dataset,
            "variant": variant,
            "matched_entities": len(group),
            "correct_names_on_matched": int(group.name_TP.sum()),
            "name_accuracy_on_matched": float(group.name_TP.mean()),
        })

    summary = metrics.copy()
    if conditional:
        summary = summary.merge(
            pd.DataFrame(conditional),
            on=["dataset", "variant"],
            how="left",
            validate="one_to_one",
        )

    guide = [
        (
            "Comparison",
            "One row per ground-truth product. Reference values are shown on the left, "
            "with the name and price produced by each method on the right. Filter by "
            "source and the __match_status / __name_ok / __price_ok fields.",
        ),
        (
            "Unmatched predictions",
            "Predictions with no corresponding ground-truth entity. They are not "
            "attached to another product and are counted as false positives (FP).",
        ),
        (
            "All results",
            "Detailed raw results in long format: one row per method and "
            "entity/prediction, including errors and identifiers.",
        ),
        (
            "Metrics",
            "name_F1 measures quality over the full corpus, including missing and "
            "unmatched predictions. name_accuracy_on_matched is the proportion of "
            "correct names only among products with a matched URL; it does not replace F1.",
        ),
        (
            "Names",
            f"name_ok: SequenceMatcher applied to lowercased strings with threshold "
            f"{NAME_THRESHOLD}. A human-readable name does not by itself imply a match "
            "with the ground-truth name.",
        ),
        (
            "Price",
            f"price_ok: relative tolerance {PRICE_REL_TOL:.0%}. "
            "gt_price_state=absent means that the absence of a price was explicitly "
            "confirmed; unannotated means that the price was not annotated. "
            "An empty field is not equivalent to the numeric value 0.",
        ),
        (
            "Statuses",
            "matched_gt — URL matched the ground truth; missing_prediction — the product "
            "was missed; unmatched_prediction — an extra prediction. An empty method "
            "field means that the method produced no result for that dataset.",
        ),
        (
            "Dataset linkage",
            "Product key: (dataset, entity_id); page key: (dataset, page_id). "
            "benchmark_run_id is stored separately for each method. gt_id identifies "
            "the original ground-truth row.",
        ),
        (
            "Pages",
            "This sheet shows empty, excluded, and missing pages. They are not converted "
            "into artificial product entities on the Comparison sheet.",
        ),
        (
            "Safe display",
            "Text is written literally, including strings beginning with =. Invalid XML "
            "characters are removed only from the Excel representation, and long cells "
            "are limited to 32,767 characters. Complete data remain available in the CSV files.",
        ),
        (
            "Raw results",
            "The Excel export does not modify extraction results or recompute stored "
            "metrics. Formatting highlights FALSE values and missing entries; filters "
            "are available in the headers.",
        ),
    ]

    sheets = [
        ("Instructions", pd.DataFrame(guide, columns=["Section", "Description"])),
        ("Metrics", summary),
        ("Comparison", comparison),
        ("UnmatchedPredictions", extras),
        ("AllResults", evaluation),
        ("Pages", pages if pages is not None else pd.DataFrame()),
        ("Manifest", manifest if manifest is not None else pd.DataFrame()),
        (
            "Provenance",
            pd.DataFrame(
                [
                    {
                        "key": k,
                        "value": (
                            json.dumps(v, ensure_ascii=False, default=str)
                            if isinstance(v, (dict, list, tuple))
                            else v
                        ),
                    }
                    for k, v in (provenance or {}).items()
                ],
                columns=["key", "value"],
            ),
        ),
    ]

    workbook = Workbook()
    workbook.remove(workbook.active)
    red = PatternFill("solid", fgColor="FCE4D6")

    for title, frame in sheets:
        # Split only if an actual Excel sheet limit is reached; never silently discard rows.
        chunk_size = 1_048_575
        for chunk_index, start in enumerate(range(0, max(1, len(frame)), chunk_size)):
            sheet = workbook.create_sheet(
                title if chunk_index == 0 else f"{title}_{chunk_index + 1}"
            )
            sheet.sheet_view.zoomScale = 85
            sheet.freeze_panes = "E2" if title == "Comparison" else "A2"

            columns = list(frame.columns)
            if not columns:
                sheet.append(["No data"])
                continue

            sheet.append(columns)

            for row_index, values in enumerate(
                frame.iloc[start:start + chunk_size].itertuples(index=False, name=None),
                2,
            ):
                cleaned = []
                for value in values:
                    if isinstance(value, (dict, list, tuple)):
                        value = json.dumps(value, ensure_ascii=False, default=str)
                    elif _is_blank(value):
                        value = None
                    elif hasattr(value, "item"):
                        value = value.item()

                    if isinstance(value, str):
                        value = ILLEGAL_CHARACTERS_RE.sub("", value)[:32767]
                    cleaned.append(value)

                sheet.append(cleaned)

                for column_index in range(1, len(columns) + 1):
                    cell = sheet.cell(row_index, column_index)
                    if isinstance(cell.value, str):
                        cell.data_type = "s"
                    cell.alignment = Alignment(vertical="top", wrap_text=True)

                sheet.row_dimensions[row_index].height = 45

            sheet.auto_filter.ref = sheet.dimensions

            for i, column in enumerate(columns, 1):
                cell = sheet.cell(1, i)
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="24476A")
                cell.alignment = Alignment(wrap_text=True, vertical="top")

                key = str(column)
                width = 20

                if key.endswith(("gt_name", "pred_name")):
                    width = 55
                elif key in {"Description", "value"}:
                    width = 115
                elif key.endswith(("notes", "error_labels")):
                    width = 44
                elif "url" in key or key.endswith(
                    ("entity_id", "page_id", "benchmark_run_id")
                ):
                    width = 32
                elif "price" in key or "similarity" in key:
                    width = 17

                letter = get_column_letter(i)
                sheet.column_dimensions[letter].width = width

                if sheet.max_row > 1 and key.endswith(
                    ("name_ok", "price_ok", "price_presence_ok")
                ):
                    sheet.conditional_formatting.add(
                        f"{letter}2:{letter}{sheet.max_row}",
                        CellIsRule(operator="equal", formula=["FALSE"], fill=red),
                    )

                if sheet.max_row > 1 and key.endswith("match_status"):
                    sheet.conditional_formatting.add(
                        f"{letter}2:{letter}{sheet.max_row}",
                        CellIsRule(
                            operator="equal",
                            formula=['"missing_prediction"'],
                            fill=red,
                        ),
                    )

            sheet.row_dimensions[1].height = 44

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_name(path.stem + ".pending.xlsx")
    workbook.save(pending)
    pending.replace(path)
    return path


## 8. Self-test

In [ ]:

def _self_test():
    html_doc = r"""
    <html><head>
      <link rel="canonical" href="https://demo.ru/search?q=drill">
      <script type="application/ld+json">
      {"@context":"https://schema.org","@graph":[
        {"@type":"Product","name":"Дрель Alpha 500","url":"https://demo.ru/product/1001","offers":{"@type":"Offer","price":"10000"}},
        {"@type":"Product","name":"Дрель Beta 700","url":"https://demo.ru/product/1002","offers":{"@type":"Offer","price":"12500"}}
      ]}
      </script>
    </head><body>
      <div class="product-card"><a href="https://demo.ru/product/1001" class="name">Дрель Alpha 500</a><span class="price">10 000 ₽</span></div>
      <div class="product-card"><a href="https://demo.ru/product/1002" class="name">Дрель Beta 700</a><span class="price">12 500 ₽</span></div>
    </body></html>
    """
    soup = BeautifulSoup(html_doc, "lxml")
    assert len(extract_b1_jsonld(html_doc, soup, "https://demo.ru/search")) == 2
    assert len(extract_b2_css(html_doc, soup, "https://demo.ru/search")) == 2

    assert parse_benchmark_filename("dns-shop.ru-run-none2.html") == {
        "source": "dns-shop.ru", "run_id": "run-none2"
    }
    assert parse_benchmark_filename("bigam.ru-run-ab12-2.html") == {
        "source": "bigam.ru", "run_id": "run-ab12"
    }
    assert _name_match("Дрель Alpha 500", "дрель alpha 500", 0.85)
    assert _price_match(10050, 10000, 0.01)
    assert not _price_match(10101, 10000, 0.01)
    # Regression: pending/empty GT and negative-only pages must work with pandas 3.
    from tempfile import TemporaryDirectory

    manifest = pd.DataFrame([{
        "dataset": "self-test", "html_key": "demo.html", "html_file": "demo.html",
        "page_id": "demo", "run_id": "run-none1", "source": "demo.ru", "skip_reason": "",
    }])
    cases = [
        ([], 0, 0, 0, 0),
        ([{"html_file": "demo.html", "annotation_status": "pending"}], 0, 0, 0, 0),
        ([{"html_file": "demo.html", "is_product": 0}], 1, 0, 0, 0),
        ([{"html_file": "demo.html", "is_product": 1}], 1, 1, 1, 0),
        ([{"html_file": "demo.html", "gt_name": "Product"}], 1, 1, 0, 1),
    ]
    with TemporaryDirectory() as tmp:
        reference = Path(tmp) / "groundtruth.csv"
        for records, annotated, positive, missing_name, name_fn in cases:
            pd.DataFrame(records, columns=[
                "html_file", "annotation_status", "is_product", "gt_name", "gt_price", "product_url",
            ]).to_csv(reference, index=False)
            gt = align_groundtruth_to_manifest(read_groundtruth(reference), manifest)
            # Explicit StringDtype exercises the empty-string reduction on pandas 2 as well.
            gt["gt_name"] = gt["gt_name"].astype("string")
            audit = groundtruth_audit(gt, manifest)
            assert audit["annotated_pages"] == annotated
            assert audit["positive_entities"] == positive
            assert audit["positive_missing_name_gt"] == missing_name
            assert audit["positive_missing_price_gt"] == positive
            assert audit["positive_missing_product_url_gt"] == positive
            _, metrics = evaluate_predictions(pd.DataFrame(), gt, manifest)
            assert metrics["name_FN"].eq(name_fn).all()
            assert metrics["price_FN"].eq(0).all()
        pd.DataFrame([{"html_file": "demo.html", "is_product": 1,
                       "gt_name": "Product", "gt_price": 0}]).to_csv(reference, index=False)
        industrial_gt = read_groundtruth(reference, zero_price_is_missing=True)
        assert industrial_gt["gt_price"].isna().all()
        assert read_groundtruth(reference)["gt_price"].eq(0).all()
        _, metrics = evaluate_predictions(pd.DataFrame(), industrial_gt, manifest)
        assert metrics["price_FN"].eq(0).all()
    assert all(_is_blank(value) for value in (None, float("nan"), pd.NA, "", "  "))
    return "OK"


print("Self-test:", _self_test())

# Reporting-only adapters (separate from the frozen cache contract)
REPORTING_CODE_SHA256 = '9e45b59978320713d494504cdce6add29c0341ba6d2a6f21467c8063814da731'
"""Reporting-only adapters, embedded after the frozen benchmark definitions.

They do not alter extraction, prompts, scoring or cache signatures. The adapter's
own hash is recorded separately from the frozen extraction/evaluation contract.
Pandas propagates attrs by deepcopy to each group/column: keep bulky response
ledgers on the original frame, never on the disposable deduplication workspace.
"""
import functools as _reporting_functools
import threading as _reporting_threading
import time as _reporting_time


def _reporting_original(function):
    return getattr(function, '_reporting_original_function', function)


def _reporting_collapse_adapter(function):
    original = _reporting_original(function)

    @_reporting_functools.wraps(original)
    def collapse(pred):
        # DataFrame construction preserves values/index/dtypes without copying attrs.
        # Do not clear pred.attrs: exports still require logs, records and provenance.
        if pred.empty:
            return original(pred)
        return original(pd.DataFrame(pred, copy=True))

    collapse._reporting_original_function = original
    return collapse


def _reporting_provenance():
    return dict(version=1, implementation_sha256=REPORTING_CODE_SHA256,
                source='scripts/benchmark_reporting.py; embedded in notebook',
                scope='attrs-free deduplication workspace and reporting progress only; frozen extraction/scoring/cache contract unchanged')


def _reporting_event(stage, scope, status, elapsed=None, error_type=None):
    event = dict(benchmark_run_id=BENCHMARK_RUN_ID, stage=stage, scope=scope, status=status,
                 at_utc=datetime.now(timezone.utc).isoformat(), elapsed_seconds=elapsed,
                 reporting_implementation_sha256=REPORTING_CODE_SHA256)
    if error_type:
        event['error_type'] = error_type
    folder = Path(OUTPUT_DIR)
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / 'reporting_progress.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False) + '\n')


def _reporting_stage_adapter(function, stage, scope_function, annotate_metadata=False):
    original = _reporting_original(function)

    @_reporting_functools.wraps(original)
    def timed(*args, **kwargs):
        scope = str(scope_function(args, kwargs))
        started = _reporting_time.perf_counter()
        stopped = _reporting_threading.Event()
        print(f'[{scope}] {stage}: started', flush=True)
        _reporting_event(stage, scope, 'started')

        def heartbeat():
            while not stopped.wait(30):
                print(f'[{scope}] {stage}: still running ({_reporting_time.perf_counter() - started:.0f}s)', flush=True)

        worker = _reporting_threading.Thread(target=heartbeat, daemon=True)
        worker.start()
        try:
            result = original(*args, **kwargs)
            if annotate_metadata:
                path = result['run_metadata']
                metadata = json.loads(path.read_text(encoding='utf-8'))
                metadata['reporting_implementation'] = _reporting_provenance()
                path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
            elapsed = _reporting_time.perf_counter() - started
            _reporting_event(stage, scope, 'complete', elapsed)
            print(f'[{scope}] {stage}: complete ({elapsed:.2f}s)', flush=True)
            return result
        except BaseException as exc:
            _reporting_event(stage, scope, 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed',
                             _reporting_time.perf_counter() - started, type(exc).__name__)
            raise
        finally:
            stopped.set()
            worker.join(timeout=1)

    timed._reporting_original_function = original
    return timed


def _reporting_manifest_scope(args, kwargs):
    manifest = args[2] if len(args) > 2 else kwargs['manifest']
    return str(manifest.dataset.iloc[0]) if len(manifest) else 'empty dataset'


def _reporting_dataset_scope(args, kwargs):
    return args[0] if args else kwargs.get('dataset', kwargs.get('dataset_name', 'dataset'))


def _reporting_workbook_scope(args, kwargs):
    return Path(args[0] if args else kwargs['path']).name


collapse_prediction_keys = _reporting_collapse_adapter(collapse_prediction_keys)
evaluate_predictions = _reporting_stage_adapter(evaluate_predictions, 'Evaluate predictions', _reporting_manifest_scope)
export_dataset_outputs = _reporting_stage_adapter(export_dataset_outputs, 'Export CSV / Excel', _reporting_dataset_scope, True)
export_review_workbook = _reporting_stage_adapter(export_review_workbook, 'Write workbook', _reporting_workbook_scope)
if 'export_llm_analysis' in globals():
    export_llm_analysis = _reporting_stage_adapter(export_llm_analysis, 'Paper metrics / cost analysis', _reporting_dataset_scope)
if 'llm_compare_determined' in globals():
    llm_compare_determined = _reporting_stage_adapter(llm_compare_determined, 'Compare determined baseline', _reporting_dataset_scope)


## 9. Run the original baseline and export all analytics

Run the cells in order. By default the exact embedded original parser runs on all three datasets.
Use `RUN_DATASETS = ("industrial",)` to reproduce the reference industrial comparison only.
Keep both `original_fast` and `original_fast_query` enabled to reproduce both original rows in Excel.

`RESUME_EXTRACTION = True` resumes interrupted work only for identical HTML, requests, parser and Node
version. Cached timings are left empty. Normally keep it False for a fresh reproducible run.
The query mode shares the original extraction; its elapsed time includes extraction + query filtering.
Do not sum shared timings across the two modes.

Original response JSONL, code/dependency hashes, Node/Python versions, predictions, metrics, page/entity
analytics and Excel are saved under `baseline_results/determined/<dataset>/`.
The combined Excel is `baseline_results/determined/baseline_review_all.xlsx`.


In [11]:

results = {}
all_metrics_parts = []
all_eval_parts = []
all_raw_parts = []
all_page_parts = []

selected_datasets = list(DATASETS) if RUN_DATASETS is None else list(RUN_DATASETS)
if not selected_datasets or len(set(selected_datasets)) != len(selected_datasets) or set(selected_datasets) - set(DATASETS):
    raise ValueError(f"RUN_DATASETS must contain distinct configured names: {list(DATASETS)}")
combined_sources = []

for dataset_name in selected_datasets:
    cfg = DATASETS[dataset_name]
    dataset_root = Path(cfg["root"])
    html_dir = Path(cfg["html_dir"])
    reference_file = cfg.get("reference_file")
    if reference_file is not None:
        reference_file = Path(reference_file)
    else:
        reference_file = discover_dataset_reference(dataset_root)

    print("\n" + "=" * 80)
    print(dataset_name.upper())
    print("HTML_DIR =", html_dir)
    print("REFERENCE =", reference_file or "<not found>")

    manifest = build_html_manifest(dataset_name, html_dir, reference_file)
    print(f"HTML files: {len(manifest)}; usable: {(manifest['skip_reason'] == '').sum() if len(manifest) else 0}")

    raw_pred = run_baselines(manifest, VARIANTS)
    print(f"Raw predictions: {len(raw_pred)}")

    gt = None
    audit = None
    evaluation = None
    metrics = None

    if reference_file is not None and Path(reference_file).exists():
        gt = read_groundtruth(Path(reference_file), "GroundTruth",
                              zero_price_is_missing=(dataset_name == "industrial"),
                              annotation_policy=ANNOTATION_POLICY)

        gt = align_groundtruth_to_manifest(gt, manifest)
        audit = groundtruth_audit(gt, manifest)
        print("GroundTruth audit:")
        print(json.dumps(audit, ensure_ascii=False, indent=2))

        if FAIL_ON_INCOMPLETE_GT and not audit["paper_ready_closed_world"]:
            raise RuntimeError(f"{dataset_name}: GroundTruth is not paper-ready.")

        if audit["annotated_pages"] > 0:
            evaluation, metrics = evaluate_predictions(raw_pred, gt, manifest, VARIANTS)
            display(metrics[METRIC_COLUMNS])
            all_metrics_parts.append(metrics)
            all_eval_parts.append(evaluation)
        else:
            print("No annotated pages yet; metrics skipped.")
    else:
        print("GroundTruth not found; extraction completed, metrics skipped.")

    paths = export_dataset_outputs(
        dataset_name, manifest, raw_pred, gt, audit, evaluation, metrics
    )
    for label, path in paths.items():
        print(f"  {label}: {path}")
    all_page_parts.append(pd.read_csv(paths["page_results"]))

    all_raw_parts.append(raw_pred)
    results[dataset_name] = {
        "reference_file": str(reference_file) if reference_file else None,
        "manifest": manifest,
        "raw_predictions": raw_pred,
        "groundtruth": gt,
        "audit": audit,
        "evaluation": evaluation,
        "metrics": metrics,
        "paths": paths,
    }

# Reuse unselected datasets only if their inputs, settings and CSV fingerprints match.
for dataset_name, cfg in DATASETS.items():
    if dataset_name in selected_datasets:
        combined_sources.append({"dataset": dataset_name, "status": "computed", "benchmark_run_id": BENCHMARK_RUN_ID})
        continue
    saved, provenance = load_verified_saved_results(dataset_name, cfg)
    combined_sources.append(provenance)
    if saved is None:
        print(f"Combined tables omit {dataset_name}: {provenance['status']}. Select it in RUN_DATASETS to refresh.")
        continue
    all_metrics_parts.append(saved["metrics"])
    all_eval_parts.append(saved["predictions"])
    all_raw_parts.append(saved["raw_predictions"])
    all_page_parts.append(saved["page_results"])
    print(f"Reusing verified {dataset_name} results from {provenance['benchmark_run_id']}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "baseline_combined_manifest.json").write_text(
    json.dumps({"aggregation_run_id": BENCHMARK_RUN_ID, "datasets": combined_sources}, ensure_ascii=False, indent=2), encoding="utf-8")

all_raw_predictions = pd.concat(all_raw_parts, ignore_index=True) if all_raw_parts else pd.DataFrame()
all_metrics = pd.concat(all_metrics_parts, ignore_index=True) if all_metrics_parts else pd.DataFrame()
all_evaluation = pd.concat(all_eval_parts, ignore_index=True) if all_eval_parts else pd.DataFrame()

all_raw_predictions.to_csv(OUTPUT_DIR / "baseline_raw_predictions_all.csv", index=False, encoding="utf-8-sig")
all_evaluation.reindex(columns=EXPORT_EVALUATION_COLUMNS).to_csv(
    OUTPUT_DIR / "baseline_predictions_all.csv", index=False, encoding="utf-8-sig")
all_pages = pd.concat(all_page_parts, ignore_index=True) if all_page_parts else pd.DataFrame(columns=PAGE_RESULT_COLUMNS)
all_pages.to_csv(OUTPUT_DIR / "baseline_page_results_all.csv", index=False, encoding="utf-8-sig")
if not all_metrics.empty:
    all_metrics[METRIC_COLUMNS].to_csv(OUTPUT_DIR / "baseline_metrics_all.csv", index=False, encoding="utf-8-sig")
    # Display rounding needs no optional Jinja2 dependency; CSV keeps full precision.
    display(paper_table(all_metrics).round(3))
else:
    pd.DataFrame(columns=METRIC_COLUMNS).to_csv(OUTPUT_DIR / "baseline_metrics_all.csv", index=False, encoding="utf-8-sig")

review_path = export_review_workbook(
    OUTPUT_DIR / "baseline_review_all.xlsx", all_evaluation, all_metrics, all_pages,
    provenance={"datasets": combined_sources})
print("Excel review:", review_path)



SYNTHETIC
HTML_DIR = D:\Pet_Codes\science\article-price-tracker\synthetic\html
REFERENCE = D:\Pet_Codes\science\article-price-tracker\synthetic\synthetic_ground_truth.csv
HTML files: 20; usable: 20
[synthetic] Original parser: 0/20 saved checkpoints
[synthetic] Original parser: 10/20; ok
[synthetic] Original parser: 20/20; ok
Raw predictions: 504
GroundTruth audit:
{
  "reference_rows": 420,
  "manifest_pages": 20,
  "available_manifest_pages": 20,
  "annotated_pages": 20,
  "unannotated_manifest_pages": 0,
  "reference_pages_not_in_html_dir": 0,
  "positive_entities": 420,
  "negative_or_empty_rows": 0,
  "positive_missing_name_gt": 0,
  "positive_missing_price_gt": 0,
  "positive_confirmed_absent_price_gt": 0,
  "positive_unannotated_price_gt": 0,
  "positive_missing_product_url_gt": 0,
  "duplicate_positive_page_url_keys": 0,
  "missing_html_files": [],
  "html_hash_mismatches": [],
  "excluded_gt_pages": 0,
  "annotation_policy": "labeled",
  "annotation_status_counts": {
    "": 

,benchmark_run_id,dataset,variant,n_pages,n,n_priced,n_price_absent,n_price_unannotated,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,unexpected_price_FP,name_TP,name_FP,name_FN,name_P,name_R,name_F1,price_TP,price_FP,price_FN,price_P,price_R,price_F1
0,20261003T090603Z-d84a171c,synthetic,original_fast,20,420,420,0,0,252,0,168,1.0,0.6,0.75,0,252,0,168,1.0,0.6,0.75,252,0,168,1.0,0.6,0.75
1,20261003T090603Z-d84a171c,synthetic,original_fast_query,20,420,420,0,0,252,0,168,1.0,0.6,0.75,0,252,0,168,1.0,0.6,0.75,252,0,168,1.0,0.6,0.75


  manifest: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\html_manifest.csv
  raw_predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_raw_predictions.csv
  groundtruth_loaded: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\groundtruth_loaded.csv
  audit: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\groundtruth_audit.json
  predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_predictions.csv
  metrics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_metrics.csv
  original_responses: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\original_parser_responses.jsonl
  entity_analytics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\synthetic\baseline_entity_analytics.csv
  page_results: D:\Pe

,benchmark_run_id,dataset,variant,n_pages,n,n_priced,n_price_absent,n_price_unannotated,entity_TP,entity_FP,entity_FN,entity_P,entity_R,entity_F1,unexpected_price_FP,name_TP,name_FP,name_FN,name_P,name_R,name_F1,price_TP,price_FP,price_FN,price_P,price_R,price_F1
0,20261003T090603Z-d84a171c,robustness,original_fast,200,4200,4200,0,0,2394,0,1806,1.0,0.57,0.726115,0,2394,0,1806,1.0,0.57,0.726115,2394,0,1806,1.0,0.57,0.726115
1,20261003T090603Z-d84a171c,robustness,original_fast_query,200,4200,4200,0,0,2394,0,1806,1.0,0.57,0.726115,0,2394,0,1806,1.0,0.57,0.726115,2394,0,1806,1.0,0.57,0.726115


  manifest: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\html_manifest.csv
  raw_predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_raw_predictions.csv
  groundtruth_loaded: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\groundtruth_loaded.csv
  audit: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\groundtruth_audit.json
  predictions: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_predictions.csv
  metrics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_metrics.csv
  original_responses: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\original_parser_responses.jsonl
  entity_analytics: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\robustness\baseline_entity_analytics.csv
  page_result

,Dataset,Method,n_pages,n,name_P,name_R,name_F1,price_P,price_R,price_F1
0,synthetic,Original fast-renderer,20,420,1.000,0.600,0.750,1.000,0.600,0.750
1,synthetic,Original + query filter,20,420,1.000,0.600,0.750,1.000,0.600,0.750
2,robustness,Original fast-renderer,200,4200,1.000,0.570,0.726,1.000,0.570,0.726
3,robustness,Original + query filter,200,4200,1.000,0.570,0.726,1.000,0.570,0.726
4,industrial,Original fast-renderer,239,6317,0.953,0.916,0.935,0.899,0.887,0.893
5,industrial,Original + query filter,239,6317,0.938,0.490,0.643,0.873,0.468,0.609


Excel review: D:\Pet_Codes\science\article-price-tracker\baseline_results\determined\baseline_review_all.xlsx


## 10. Scope of the paper baseline

This is the original deterministic HTML extraction mechanism, frozen in the notebook.
It includes built-in supplier profiles, but no live browser, external source configurations, database
or LLM fallback. Names/prices are evaluated with the same GT, URL normalization, thresholds and
absence-of-price policy as `original_comparison.xlsx`. LLM and hybrid runs belong in separate result folders.
